# step5_rerank_mmr

**Owner:** P6  
**Purpose:** Tune MMR on the tuning split, then evaluate the fixed choice on held-out data.  
**Inputs:** Complete processed candidates, the chosen model's scores for both splits, article embeddings with their ID mapping, and article-title metadata for illustrative examples.  
**Outputs:** Full evaluation score Parquets for each lambda under `config.SCORES_DIR`; tuning/evaluation result CSVs under `config.OUTPUT_DIR`, detailed evaluation summaries in `step5_evaluation_summary`, and figures, examples, conclusions, and metadata in `step5_report`.

Run either the Colab setup or the local VS Code setup, then Sections 1-4 in order in one kernel session. Fill the two embedding-export paths in 1.1 and choose the final relevance-loss budget in 2.4.1 before the complete run. Use the same upstream model and encoder versions for both splits. After changing inputs or experiment settings, restart the kernel and run from the beginning.

In [ ]:
# --- Setup: run this first, every session ---
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/BT4222-Project-Group-4--MIND && git clone -q https://github.com/Fdiannz/BT4222-Project-Group-4--MIND.git /content/BT4222-Project-Group-4--MIND
import sys; sys.path.insert(0, '/content/BT4222-Project-Group-4--MIND')

import numpy as np, pandas as pd
# Import plotting tools once for the later presentation and export sections.
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.ticker import PercentFormatter
import config
print("Data folder:", config.DATA_DIR)

### Alternative setup: local VS Code environment

**Run this cell instead of the Colab setup above.** Select a Python notebook kernel with `numpy`, `pandas`, and `matplotlib` installed. Reading processed Parquet files later also requires a Parquet engine such as `pyarrow`. Google Drive for desktop must be running and the shared shortcut accessible.

This imports the local checkout and redirects the imported `config` module's paths **for the current kernel only**. It does not edit `config.py`, create folders, or load the datasets into memory. Restart the kernel when switching setups or source checkouts. Later code should use the configured paths.

This setup cell is temporary, and will be removed before the final submission. Its purpose is to serve as a way to import data from the shared drive without using colab.

In [ ]:
# --- Local VS Code setup: run this INSTEAD OF the Colab setup above. ---
# Import this computer's project helpers and configure the shared Drive paths.
# This cell does not clone repositories, modify project files, or load datasets.
from pathlib import Path
import sys

# Use absolute paths so the notebook works from any working directory.
LOCAL_REPO_DIR = Path(r"C:\Users\brchu\Documents\BT4222 Project\BT4222-Project-Group-4--MIND")
LOCAL_SHARED_DIR = Path(r"G:\.shortcut-targets-by-id\1D4Db1s7DJ5PVK9CFiV-pklpCW5dr1uOM\BT4222_Project")

# Fail early if the source checkout or Google Drive shortcut is unavailable.
if not (LOCAL_REPO_DIR / "config.py").is_file():
    raise FileNotFoundError(f"Project source folder is unavailable: {LOCAL_REPO_DIR}")
if not (LOCAL_SHARED_DIR / "data" / "processed").is_dir():
    raise FileNotFoundError(
        f"Processed data folder is unavailable: {LOCAL_SHARED_DIR / 'data' / 'processed'}. "
        "Check that Google Drive for desktop is running and the shortcut is accessible."
    )

# Give this checkout priority when Python imports the shared project modules.
# Restart the kernel first if another checkout's modules were already imported.
repo_path = str(LOCAL_REPO_DIR)
if repo_path in sys.path:
    sys.path.remove(repo_path)
sys.path.insert(0, repo_path)

# Import the numerical libraries and the team's shared configuration.
import numpy as np
import pandas as pd
# Import plotting tools once for the later presentation and export sections.
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.ticker import PercentFormatter
import config

# Change runtime paths only; preserve the team's seed, splits, and MMR settings.
# Forward slashes work on Windows and match the existing configuration style.
config.PROJECT_DIR = LOCAL_SHARED_DIR.as_posix()
config.DATA_DIR = f"{config.PROJECT_DIR}/data"
config.RAW_DIR = f"{config.DATA_DIR}/raw"
config.PROCESSED_DIR = f"{config.DATA_DIR}/processed"
config.OUTPUT_DIR = f"{config.PROJECT_DIR}/outputs"
config.SCORES_DIR = f"{config.DATA_DIR}/scores"
config.TRAIN_DIR = f"{config.RAW_DIR}/MINDlarge_train"
config.DEV_DIR = f"{config.RAW_DIR}/MINDlarge_dev"
config.TEST_DIR = f"{config.RAW_DIR}/MINDlarge_test"

# Print active locations for verification; scores/output folders may not exist yet.
# Actual data loading and experiment execution will happen in later sections.
print("Source folder:", LOCAL_REPO_DIR)
print("Data folder:", config.DATA_DIR)
print("Processed data:", config.PROCESSED_DIR)
print("Model scores:", config.SCORES_DIR)
print("Output folder:", config.OUTPUT_DIR)

## 1. Inputs, relevance baseline, and MMR implementation

Load complete processed candidate slates and attach the chosen model's scores. Load and normalize the article vectors, then establish a deterministic relevance-only baseline with overall and eligible-slate relevance, diversity, and cold-exposure summaries.

Implement greedy MMR using the configured `TOP_K` (currently 10). Rerank only slates with more than `TOP_K` candidates and preserve a complete candidate order for evaluation. Original model scores remain separate from final ranks. Section 1.5 checks the implementation using invented examples and an independent reference; those checks are not measured project results.

### 1.1 Input paths and settings

Use the active setup's `config` paths. The score inputs below are configured for P5's `nrms_main_v2` run: `step4_nrms_tune.parquet` and `step4_nrms_eval.parquet` under `config.SCORES_DIR`, with numeric column `score`. The selected encoder is `nrms_main_v2_best.pt:news_encoder`. Fill the two embedding export paths before the complete run. Relevance scaling is set to `"per_impression_minmax"`. `None` means a required choice or artifact has not been supplied. `TOP_K`, `SEED`, and `LAMBDA_VALUES` come from the shared configuration; retain the configured `RELEVANCE_SCALING` for evaluation. Section 1.4.1 explains the scaling conventions.

| Variable | Required value or artifact |
|---|---|
| `BASE_MODEL_NAME` | `"nrms_main_v2"`, identifying the current NRMS score producer. |
| `TUNE_SCORES_PATH`, `EVAL_SCORES_PATH` | Parquet files with one finite score for every processed candidate, keyed by `impression_id` and `news_id`. Build paths from `config.SCORES_DIR`. The evaluation file covers the labelled MIND dev evaluation split. |
| `SCORE_COLUMN` | `"score"`, the float32 column in both NRMS exports. The joined table keeps this name. |
| `NEWS_EMBEDDINGS_PATH` | Awaited export: a numeric `.npy` matrix with one 256-dimensional NRMS vector per real article, covering both splits. Set the path to the actual delivered file. |
| `NEWS_EMBEDDING_IDS_PATH` | Awaited export: unique `news_id` strings in exactly the vector-row order, stored as a 1-D Unicode `.npy` array compatible with `allow_pickle=False`. |
| `EMBEDDING_ENCODER_ID` | `"nrms_main_v2_best.pt:news_encoder"`, identifying the selected P5 encoder. The checkpoint is under `config.OUTPUT_DIR/step4_nrms/`; the pending exports must use its trained weights. |

P5's saved configuration uses title text (up to 30 tokens), a 256-dimensional news encoder, and seed 42. Its `NEWS_ROW` mapping assigns real articles rows 1 through N, with row 0 reserved for padding. The export must pair `NEWS_VECS[1:]` with the `news_id` values ordered by that mapping; padding has no real article ID and must be excluded. Both score files use string `impression_id`/`news_id` keys, matching the processed candidates. The current NRMS score row counts match their corresponding candidate files; the strict join in 1.2 still verifies actual key coverage when run.

The processed `tune_candidates.parquet` and `eval_candidates.parquet` paths are constructed from `config.PROCESSED_DIR`. Read `impression_id`, `news_id`, `label`, `cold_article`, and `category` for every candidate. The processed files also contain `time`, `prior_exposure_count`, and `subcategory`; these are not loaded because this notebook does not use them. Keep the upstream `cold_article` name and pass it explicitly to metric helpers. Candidate cleaning, cold flags, and split construction belong upstream.

Section 4.4 also reads `config.PROCESSED_DIR/news_metadata.parquet`, using its `news_id` and `title` columns to annotate examples. It defines this path as `EXAMPLE_METADATA_PATH`; titles do not affect ranking or selection.

Section 1 loads tuning inputs. Evaluation inputs are registered here and loaded in Section 3. If upstream file formats change, adapt the relevant reader in 1.2 and preserve its downstream column/vector interface.

In [ ]:
# --- 1.1a: Import shared helpers and use the active setup's paths/settings. ---
from pathlib import Path
from copy import deepcopy
from utils.metrics import attach_scores, add_rank, topk

# Both the standard Colab setup and temporary local setup configure this module.
TUNE_CANDIDATES_PATH = Path(config.PROCESSED_DIR) / "tune_candidates.parquet"
EVAL_CANDIDATES_PATH = Path(config.PROCESSED_DIR) / "eval_candidates.parquet"
KEY_COLUMNS = ["impression_id", "news_id"]
COLD_COLUMN = "cold_article"
# Keep every candidate row, but load only the fields consumed by reranking and evaluation.
CANDIDATE_COLUMNS = KEY_COLUMNS + ["label", COLD_COLUMN, "category"]
TOP_K, SEED = config.TOP_K, config.SEED
LAMBDA_VALUES = tuple(config.MMR_LAMBDAS)
INPUT_SPLITS = {"tune": TUNE_CANDIDATES_PATH, "eval": EVAL_CANDIDATES_PATH}
print("Top k:", TOP_K, "| Seed:", SEED, "| Lambda grid:", LAMBDA_VALUES)

In [ ]:
# --- 1.1b: Configure NRMS scores and fill the remaining embedding/settings placeholders. ---
# Build paths from config; subsequent cells do not use local Windows paths.
# Use P5's current main-model score exports for both splits.
BASE_MODEL_NAME = "nrms_main_v2"
TUNE_SCORES_PATH = Path(config.SCORES_DIR) / "step4_nrms_tune.parquet"
EVAL_SCORES_PATH = Path(config.SCORES_DIR) / "step4_nrms_eval.parquet"
SCORE_COLUMN = "score"
# P5 has not yet exported the article vectors and their aligned IDs; filenames are unknown.
NEWS_EMBEDDINGS_PATH = None
NEWS_EMBEDDING_IDS_PATH = None
# Pending vectors must come from the same selected checkpoint as the score producer.
EMBEDDING_ENCODER_ID = "nrms_main_v2_best.pt:news_encoder"
RELEVANCE_SCALING = "per_impression_minmax"  # Balance within-slate scores against cosine similarity.

# Fill only the two export paths after receiving P5's vectors and matching article IDs.
print("Model:", BASE_MODEL_NAME, "| Encoder:", EMBEDDING_ENCODER_ID)
print("Tuning scores:", TUNE_SCORES_PATH, "| Evaluation scores:", EVAL_SCORES_PATH)

### 1.2 Load processed candidates, model scores, and article vectors

Trust upstream candidate cleaning and split construction. Load complete tables without sampling, row filtering, repeated date/label/category checks, or another comparison against impression files. Check only the separate score join and vector alignment/coverage here.

#### 1.2.1 Define input readers

Project the required columns. Pandas/PyArrow report missing files, columns, or dependencies directly. The shared score helper enforces a one-to-one complete join; check finite scores once to prevent invalid MMR arithmetic.

In [ ]:
# --- 1.2.1: Read processed candidates and join separately delivered scores. ---
def load_candidate_split(split):
    """Read the full processed split; cleaning and slate construction belong upstream."""
    return pd.read_parquet(INPUT_SPLITS[split], columns=CANDIDATE_COLUMNS, engine="pyarrow")


def load_scored_split(split, candidates):
    """Attach one original score per candidate using the shared strict join."""
    path = TUNE_SCORES_PATH if split == "tune" else EVAL_SCORES_PATH
    scores = pd.read_parquet(path, columns=KEY_COLUMNS + [SCORE_COLUMN], engine="pyarrow")
    scores = scores.rename(columns={SCORE_COLUMN: "score"})
    # This score artifact is separate from the already-cleaned candidate table.
    if not np.isfinite(scores["score"].to_numpy(dtype=np.float64)).all():
        raise ValueError("Model scores must be finite.")
    return attach_scores(candidates, scores)

#### 1.2.2 Load complete tuning candidates

Read all processed tuning candidates. Counts and the short preview describe the input; they do not perform another cleaning pass.

In [ ]:
# --- 1.2.2: Load complete tuning candidates without resampling/re-cleaning. ---
tune_candidates = load_candidate_split("tune")
print("Tuning rows:", len(tune_candidates), "| Impressions:", tune_candidates["impression_id"].nunique())
print(tune_candidates.head(5).to_string(index=False))

#### 1.2.3 Attach tuning model scores

Join supplied scores on impression/article IDs and preserve them for original-score AUC. Release the duplicate unscored table after joining.

In [ ]:
# --- 1.2.3: Attach tuning scores and inspect their original range. ---
tune_scored = load_scored_split("tune", tune_candidates)
del tune_candidates  # The joined table now contains every required candidate field.
print(tune_scored[KEY_COLUMNS + ["score"]].head(5).to_string(index=False))
print("Score range:", tune_scored["score"].min(), "to", tune_scored["score"].max())

#### 1.2.4 Load and normalize article embeddings

The separate vector and ID artifacts must align row-for-row with unique article IDs. Reject nonfinite/zero vectors because cosine similarity is undefined for them. Normalize the lookup and check each split's article coverage once at its input boundary. Use P5's selected NRMS news encoder/catalog for both splits, with its padding row removed and ID mapping preserved as described in 1.1.

In [ ]:
# --- 1.2.4a: Build a keyed, normalized article-vector lookup. ---
def build_embedding_lookup(vectors, news_ids):
    """Check the new artifact's alignment and normalize vectors for cosine similarity."""
    values = np.asarray(vectors, dtype=np.float64)
    ids = np.asarray(news_ids)
    if values.ndim != 2 or ids.ndim != 1 or len(ids) != len(values) or values.size == 0:
        raise ValueError("Embeddings need a nonempty matrix and one news ID per row.")
    if pd.isna(ids).any() or pd.Index(ids).has_duplicates:
        raise ValueError("Embedding article IDs must be nonmissing and unique.")
    norms = np.linalg.norm(values, axis=1)
    if not np.isfinite(values).all() or not np.isfinite(norms).all() or (norms <= 0).any():
        raise ValueError("Cosine embeddings must be finite and nonzero.")
    return dict(zip(ids, (values / norms[:, None]).astype(np.float32)))


def require_embedding_coverage(candidates, lookup, split):
    """Check the separately supplied catalog covers every article in this split."""
    missing = pd.Index(candidates["news_id"].unique()).difference(lookup)
    if len(missing):
        raise ValueError(f"{split}: {len(missing)} articles lack vectors; examples: {missing[:5].tolist()}")
    print(f"{split}: all candidate articles have vectors.")


def load_article_embeddings(vector_path, ids_path):
    """Load upstream .npy artifacts without assuming metadata row order."""
    return build_embedding_lookup(np.load(vector_path, allow_pickle=False),
                                  np.load(ids_path, allow_pickle=False))

In [ ]:
# --- 1.2.4b: Load the article catalog and check tuning coverage once. ---

# Both splits use this catalog and the same upstream encoder.
emb_lookup = load_article_embeddings(NEWS_EMBEDDINGS_PATH, NEWS_EMBEDDING_IDS_PATH)
require_embedding_coverage(tune_scored, emb_lookup, "tune")
print("Loaded article vectors:", len(emb_lookup))

### 1.3 Relevance-only baseline

Establish the original model ranking from the complete tuning inputs loaded in 1.2. Preserve original `score` values and calculate relevance, diversity, and cold exposure before applying MMR.

Outputs are `baseline_ranked` (complete candidates and reference ranks), `baseline_slates` (impression sizes and eligibility), `baseline_top` (up to `TOP_K` articles per impression), `baseline_relevance`, and `baseline_diversity`. The complete run requires the score and embedding artifacts from 1.2. Sections 1.3.2-1.3.3 define the metric populations and unavailable-value conventions.

#### 1.3.1 Construct the original ranking and define eligibility

Rank each full impression by descending model score. Sort candidate keys into a canonical order before calling the shared seeded `add_rank()` helper. This makes identical inputs reproducible even if score-file rows arrive in a different order. MMR reuses this reference rank to break ties and to reproduce the baseline at lambda = 1.0; do not resolve ties independently inside each comparison subset.

An impression is **eligible for re-ranking only when its candidate count is greater than `TOP_K`**. Exactly-`TOP_K` and shorter slates remain in the overall relevance reference but are not eligible. All candidates remain in `baseline_ranked`; top-k extraction does not replace the full ranking used for relevance metrics.

In [ ]:
# --- 1.3.1a: Build a reproducible relevance-only reference from joined model scores. ---
def build_relevance_reference(scored, k=None, seed=None):
    """Return complete ranks, slate eligibility, and top-k lists from Section 1.2 inputs."""
    # Synthetic cases may supply their own k/seed; actual runs use shared defaults.
    k = TOP_K if k is None else k
    seed = SEED if seed is None else seed
    required = KEY_COLUMNS + ["label", COLD_COLUMN, "category", "score"]

    # Canonical ordering makes seeded tie draws independent of input row order.
    # Keep only the fields needed downstream to limit extra copies of the large table.
    canonical = scored[required].sort_values(KEY_COLUMNS, kind="stable").reset_index(drop=True)
    ranked = add_rank(canonical, "score", rank_col="baseline_rank", seed=seed)
    del canonical

    # Determine eligibility from the full slate, never from a truncated top-k list.
    slates = ranked.groupby("impression_id", sort=False).size().rename("n_candidates").to_frame()
    slates["rerank_eligible"] = slates["n_candidates"].gt(k)
    ranked["rerank_eligible"] = ranked["impression_id"].map(slates["rerank_eligible"])

    # Pre-filter by known rank before using the shared top-k helper to avoid sorting
    # millions of candidates that cannot appear in the displayed reference lists.
    top = topk(ranked.loc[ranked["baseline_rank"].le(k)], "baseline_rank", k=k)
    return ranked, slates, top

In [ ]:
# --- 1.3.1b: Build the deterministic baseline and inspect eligibility. ---
baseline_ranked, baseline_slates, baseline_top = build_relevance_reference(tune_scored, k=TOP_K, seed=SEED)
del tune_scored  # The baseline retains every field needed by ranking and metrics.
print("Eligible slates:", int(baseline_slates["rerank_eligible"].sum()))
print(baseline_top.head(10).to_string(index=False))

#### 1.3.2 Measure reference relevance overall and on eligible impressions

Report both **all impressions** and the **eligible re-ranking subset**, with cold/warm/mixed segments defined by the clicked articles' cold status. A cold segment means all clicked articles are cold, warm means all are warm, and mixed contains both. This is impression segmentation, not filtering cold candidates out of their competing slates.

Use the notebook-local raw adapter over the shared per-impression and segment functions; the shared helper API is unchanged. Compute AUC from the original model scores and label it **`model_score_auc`**. Calculate MRR and nDCG from the fixed reference order using negative rank as a higher-is-better score. This avoids the shared helper redrawing score ties when a subset is evaluated and ensures ranking metrics match the displayed lists. MMR comparisons use the same function with their final rank column while retaining original-score AUC.

For consistency with the current shared helpers, impressions with no clicks or no non-clicks are skipped for all relevance metrics, including MRR/nDCG. Counts are reported as `n_impressions` and `n_skipped`. Absent segments are shown with zero counts and unavailable metrics rather than zero performance.

The raw adapter depends on the repository's `_per_impression` helper; if P2 refactors that internal interface, update the adapter and rerun the parity/regression checks. Display rounding never changes the stored selection values.

In [ ]:
# --- 1.3.2a: Summarize an explicit ranking while retaining original-score AUC. ---
# P6 keeps full-precision aggregates; P2's default four-decimal summaries stay unchanged.
def p6_evaluate_by_segment_raw(frame, score_column, cold_column, seed):
    """Aggregate shared per-impression calculations without early rounding."""
    from utils.metrics import _per_impression, label_impression_segment
    per = _per_impression(frame, score_column, "label", "impression_id", seed)
    per["segment"] = per["impression_id"].map(label_impression_segment(frame, cold_column))
    rows = {}
    for segment in ["all", "cold", "warm", "mixed"]:
        subset = per if segment == "all" else per.loc[per["segment"].eq(segment)]
        valid = subset.loc[subset["valid"]]
        row = valid[["auc", "mrr", "ndcg@5", "ndcg@10"]].mean().to_dict()
        row.update(n_impressions=int(len(valid)), n_skipped=int(len(subset) - len(valid)))
        rows[segment] = row
    result = pd.DataFrame.from_dict(rows, orient="index")
    result[["n_impressions", "n_skipped"]] = result[["n_impressions", "n_skipped"]].astype(np.int64)
    return result


def summarize_reference_relevance(ranked, rank_col="baseline_rank", seed=None):
    """Compare all/eligible full slates without recomputing tie-breaking per subset."""
    # Read the active seed at call time; explicit caller overrides still take priority.
    seed = SEED if seed is None else seed
    segments = ["all", "cold", "warm", "mixed"]
    metric_columns = ["model_score_auc", "mrr", "ndcg@5", "ndcg@10"]
    tables = []
    for scope in ["all_impressions", "eligible_impressions"]:
        # Keep this projection small; model_score and explicit rank have distinct roles.
        selected = ranked if scope == "all_impressions" else ranked.loc[ranked["rerank_eligible"]]
        if selected.empty:
            summary = pd.DataFrame(np.nan, index=segments, columns=metric_columns)
            summary["n_impressions"] = 0
            summary["n_skipped"] = 0
        else:
            inputs = selected[["impression_id", "label", COLD_COLUMN, "score"]].copy()
            original = p6_evaluate_by_segment_raw(inputs, "score", COLD_COLUMN, seed=seed)

            # A negative final rank is unique within each impression: no new tie draw.
            inputs["_order_score"] = -selected[rank_col].to_numpy(dtype=np.float64)
            ordered = p6_evaluate_by_segment_raw(inputs, "_order_score", COLD_COLUMN, seed=seed)
            summary = ordered.drop(columns="auc")
            summary.insert(0, "model_score_auc", original["auc"])
        summary.index.name = "segment"
        tables.append(summary)
    return pd.concat(tables, keys=["all_impressions", "eligible_impressions"], names=["scope"])

In [ ]:
# --- 1.3.2b: Measure relevance using the fixed baseline order. ---

# Original scores feed AUC; the fixed order feeds MRR/nDCG.
baseline_relevance = summarize_reference_relevance(baseline_ranked)
print(baseline_relevance.to_string(float_format=lambda value: f"{value:.4f}"))

#### 1.3.3 Measure reference diversity and cold exposure

Use the notebook-local raw adapter for the shared category/cold-share populations and mean off-diagonal cosine definition. All means retain full precision; lower similarity means more semantic diversity. Report **`all_full_topk_lists`**, which includes slates with at least `TOP_K` candidates, and **`eligible_impressions`**, which includes only slates with more than `TOP_K` candidates. The former excludes shorter lists under the current shared-helper convention; exactly-`TOP_K` lists appear there even though they will not be re-ranked. Both scopes average metrics equally across lists, not across candidates.

Every output includes list counts and shorter-list exclusions. The list helper represents undefined semantic similarity as `NaN` with an explicit status. If a scope contains no full lists, return unavailable metrics and zero evaluated lists without calling a helper that expects nonempty input. Only the validated, nonzero article embeddings from 1.2 should be passed here.

In [ ]:
# --- 1.3.3a: Summarize full top-k lists with explicit scope and missing-vector status. ---
# Preserve the shared population/metric definitions while retaining unrounded means.
def p6_topk_diversity_raw(top, category_col="category", emb_lookup=None, news_col="news_id",
                          cold_col=None, group_col="impression_id", k=10):
    """Compute full-list metrics from the loaded article catalog without early rounding."""
    sizes = top.groupby(group_col, sort=False)[news_col].transform("size")
    full = top.loc[sizes.eq(k)].sort_values(group_col, kind="stable")
    result = {"unique_categories": float(full.groupby(group_col)[category_col].nunique().mean())}
    if emb_lookup is not None and k > 1:
        ids = pd.Index(full[news_col].unique())
        # Input vectors were checked at loading; normalize this compact metric matrix.
        vectors = np.stack([emb_lookup[news_id] for news_id in ids]).astype(np.float32)
        norms = np.linalg.norm(vectors, axis=1, keepdims=True)
        vectors /= norms
        positions = ids.get_indexer(full[news_col]).reshape(-1, k)
        similarity = np.empty(len(positions), dtype=np.float64)
        triangle = np.triu_indices(k, 1)
        for start in range(0, len(positions), 2048):
            block = vectors[positions[start:start + 2048]]
            cosine = np.clip(np.einsum("nkd,njd->nkj", block, block), -1.0, 1.0)
            similarity[start:start + 2048] = cosine[:, triangle[0], triangle[1]].mean(axis=1, dtype=np.float64)
        result["intra_list_sim"] = float(similarity.mean())
    if cold_col is not None:
        result["cold_share"] = float(full.groupby(group_col)[cold_col].mean().mean())
    result["n_lists"] = int(full[group_col].nunique())
    result["n_short_lists_excluded"] = int(top[group_col].nunique() - result["n_lists"])
    return result


def summarize_reference_diversity(top, lookup=None, k=None):
    """Use the shared full-list definition and handle empty scopes explicitly."""
    # Resolve k from current settings, rather than retaining the value at function definition.
    k = TOP_K if k is None else k
    rows = {}
    for scope in ["all_full_topk_lists", "eligible_impressions"]:
        selected = top if scope == "all_full_topk_lists" else top.loc[top["rerank_eligible"]]
        sizes = selected.groupby("impression_id", sort=False).size()
        n_full = int(sizes.eq(k).sum())
        if n_full == 0:
            result = {
                "unique_categories": np.nan, "intra_list_sim": np.nan, "cold_share": np.nan,
                "n_lists": 0, "n_short_lists_excluded": int(len(sizes)),
            }
            status = "NO_FULL_LISTS"
        else:
            result = p6_topk_diversity_raw(selected, category_col="category", emb_lookup=lookup,
                                    cold_col=COLD_COLUMN, k=k)
            if lookup is None:
                result["intra_list_sim"] = np.nan
                status = "PENDING_EMBEDDINGS"
            elif k == 1:
                result["intra_list_sim"] = np.nan
                status = "UNDEFINED_SINGLE_ITEM_LIST"
            else:
                status = "AVAILABLE"
        result["n_input_impressions"] = int(len(sizes))
        result["semantic_status"] = status
        rows[scope] = result
    result_table = pd.DataFrame.from_dict(rows, orient="index")
    result_table.index.name = "scope"
    return result_table

In [ ]:
# --- 1.3.3b: Measure baseline diversity and cold exposure. ---

# Full-list metrics exclude short lists and retain their population counts.
baseline_diversity = summarize_reference_diversity(baseline_top, lookup=emb_lookup, k=TOP_K)
print(baseline_diversity.to_string(float_format=lambda value: f"{value:.4f}"))

### 1.4 Greedy MMR implementation

Implement Maximal Marginal Relevance as a reusable function over the validated reference ranking and article vectors. At each selection after the first, maximize:

`lambda * relevance - (1 - lambda) * max(cosine similarity to any already-selected article)`

Only slates with more than `TOP_K` candidates are re-ranked. Seed each eligible list with its highest-ranked baseline article, even at lambda = 0. Choose subsequent articles using the MMR objective; break numerical objective ties using the existing baseline order and the tolerance defined in 1.4.1. After selecting `TOP_K` articles, append unselected articles in baseline order. This produces a complete permutation, preserves the candidate set and original scores, and gives downstream metrics an explicit final order.

Cosine similarity uses its signed range [-1, 1], so negative similarity can favor dissimilar articles. There is no hard similarity cutoff, category cap, or cold-item bonus. The algorithm does not use click labels or cold flags to select articles. Lambda = 1 takes an explicit identity path to reproduce baseline ranks exactly, including original-score ties. This section defines the algorithm and checks that identity before tuning; the lambda sweep and selection remain in Section 2.

#### 1.4.1 Relevance scaling and numerical conventions

The original model `score` remains untouched. Scaling affects only the temporary relevance term used within an eligible impression. Two methods are supported:

- **`"none"`**: use the original finite scores. This preserves their scale; similarity can have little effect if scores have a much larger magnitude.
- **`"per_impression_minmax"`**: map each full slate's minimum/maximum score to 0/1. Equal-score slates receive all zeros because relevance cannot distinguish their candidates. This changes how score gaps balance similarity and is an experimental choice, not a probability calibration.

`RELEVANCE_SCALING` in 1.1 is set to `"per_impression_minmax"`. Use and record the same convention for tuning and held-out evaluation. Lambda = 1 does not need scaling or similarity calculations and returns the original order exactly. For article selection, `MMR_COSINE_TIE_ATOL = 1e-7` defines a small absolute cosine tie band. The MMR objective band is `(1 - lambda) * MMR_COSINE_TIE_ATOL`: select the lowest baseline rank among candidates within that band of the largest objective. This absorbs float32 matrix-row accumulation differences for identical vectors without using a relative tolerance that grows with raw score magnitude. Differences outside the band retain their objective ordering. The convention is fixed before tuning and recorded with the experiment; it is distinct from the policy's lambda-selection tie rule.

In [ ]:
# --- 1.4.1: Validate lambda and scale a single slate's temporary relevance values. ---
def validate_mmr_lambda(value):
    """Keep the parameter within MMR's mathematical domain."""
    if not np.isfinite(value) or not 0 <= value <= 1:
        raise ValueError("MMR lambda must be finite and in [0, 1].")
    return float(value)


SUPPORTED_RELEVANCE_SCALING = ("none", "per_impression_minmax")
# One small absolute cosine band absorbs float32 accumulation noise for numerical ties.
# Its contribution to an MMR objective is scaled by that objective's diversity weight.
MMR_COSINE_TIE_ATOL = 1e-7
MMR_PROGRESS_EVERY = 50000  # Print progress per eligible slate; use 0 to disable.


def scale_mmr_relevance(scores, method):
    """Scale one complete slate without altering its original model scores."""
    # run_mmr_ranks validates the method once before processing impressions.
    values = np.array(scores, dtype=np.float64, copy=True)
    if method == "none":
        return values

    # Rescale by magnitude first so max-min cannot overflow on extreme finite logits.
    # Constant scores have no relevance preference; baseline rank resolves any ties.
    magnitude = np.max(np.abs(values))
    if magnitude == 0:
        return np.zeros_like(values)
    bounded = values / magnitude
    lo, hi = bounded.min(), bounded.max()
    if hi == lo:
        return np.zeros_like(values)
    return np.clip((bounded - lo) / (hi - lo), 0.0, 1.0)


print("Configured scaling:", RELEVANCE_SCALING)
print("First item and numerical MMR ties use baseline order; lambda=1 preserves the reference.")

#### 1.4.2 Prepare arrays once for repeated MMR runs

Group candidates by impression and fixed baseline rank, then gather vectors into compact arrays. Reuse these across lambdas without repeating input validation.

In [ ]:
# --- 1.4.2: Prepare grouping and vector arrays for repeated MMR calls. ---
def prepare_mmr_inputs(reference, lookup, k=None):
    """Cache arrays/indexes from the baseline and loaded article catalog."""
    k = TOP_K if k is None else k
    scores = reference["score"].to_numpy(dtype=np.float64, copy=True)
    ranks = reference["baseline_rank"].to_numpy(dtype=np.int64)
    group_codes, _ = pd.factorize(reference["impression_id"], sort=False)
    sizes = np.bincount(group_codes)
    starts = np.r_[0, np.cumsum(sizes)[:-1]]
    order = np.lexsort((ranks, group_codes))

    # Gather once; normalization also supports raw-vector synthetic examples.
    news_ids = pd.Index(reference["news_id"].unique())
    matrix = np.stack([lookup[news_id] for news_id in news_ids]).astype(np.float64)
    matrix = (matrix / np.linalg.norm(matrix, axis=1, keepdims=True)).astype(np.float32)
    prepared = {
        "reference": reference, "k": int(k), "scores": scores, "baseline_ranks": ranks,
        "order": order, "starts": starts, "ends": starts + sizes,
        "eligible_groups": sizes > k,
        "article_index": news_ids.get_indexer(reference["news_id"]).astype(np.int32),
        "embeddings": matrix,
    }
    return prepared

#### 1.4.3 Greedy selection and complete final order

For each eligible slate, work in baseline order, select the first article, then maintain each remaining candidate's maximum cosine similarity to the selected set. Update this maximum with one matrix-vector product per newly selected article, rather than constructing a full slate-by-slate similarity matrix. The temporary computation scales with slate length, vector dimension, and `TOP_K`.

`run_mmr_ranks()` returns final 1-based ranks aligned to the prepared reference rows. `apply_mmr()` wraps those ranks in a scored DataFrame with **`rerank_rank`** and **`rerank_score = -rerank_rank`**, preserving `score` and `baseline_rank`. Use `summarize_reference_relevance(result, rank_col="rerank_rank")` for relevance and extract top lists using `rerank_rank` for diversity. Do not sort the MMR selection-objective values after the greedy procedure.

Process one lambda at a time when sweeping, summarize its results, and release the candidate-level result before proceeding to the next setting. Keeping a full DataFrame for every lambda would consume unnecessary memory.

In [ ]:
# --- 1.4.3a: Select a single eligible slate using incremental maximum similarity. ---
def _greedy_mmr_order(relevance, vectors, k, lambda_value):
    """Internal kernel: inputs are already sorted by baseline rank."""
    n = len(relevance)
    # Bound the tie band by cosine precision, not the magnitude of raw model scores.
    objective_tie_atol = (1.0 - lambda_value) * MMR_COSINE_TIE_ATOL
    selected = np.empty(k, dtype=np.int64)
    chosen = np.zeros(n, dtype=bool)

    # Seed with the baseline-best article: no selected set exists for a penalty yet.
    selected[0] = 0
    chosen[0] = True
    max_similarity = np.clip(vectors @ vectors[0], -1.0, 1.0).astype(np.float64)
    for step in range(1, k):
        objectives = lambda_value * relevance - (1.0 - lambda_value) * max_similarity
        objectives[chosen] = -np.inf

        # Identical vectors can differ by float32 round-off between matrix rows.
        # Among numerically tied objectives, take the first position in baseline order.
        best_objective = objectives.max()
        winner = int(np.flatnonzero(objectives >= best_objective - objective_tie_atol)[0])
        selected[step] = winner
        chosen[winner] = True
        if step < k - 1:
            new_similarity = np.clip(vectors @ vectors[winner], -1.0, 1.0)
            max_similarity = np.maximum(max_similarity, new_similarity)

    # Leave all unselected candidates in baseline order after the selected top-k.
    return np.r_[selected, np.flatnonzero(~chosen)]


def run_mmr_ranks(prepared, lambda_value, scaling=None, progress_every=0):
    """Return final ranks aligned to prepared reference rows; never modify inputs."""
    weight = validate_mmr_lambda(lambda_value)
    if scaling is not None and scaling not in SUPPORTED_RELEVANCE_SCALING:
        raise ValueError(f"Unknown relevance scaling: {scaling!r}.")
    result = prepared["baseline_ranks"].copy()

    # Preserve original order exactly even if score scaling would collapse tiny gaps.
    if weight == 1.0:
        return result
    if scaling is None:
        raise ValueError("Set RELEVANCE_SCALING explicitly before running lambda < 1.")

    # Short/equal-size slates retain their original ranks; only eligible groups run.
    eligible = np.flatnonzero(prepared["eligible_groups"])
    for completed, group in enumerate(eligible, start=1):
        start, end = prepared["starts"][group], prepared["ends"][group]
        positions = prepared["order"][start:end]
        relevance = scale_mmr_relevance(prepared["scores"][positions], scaling)
        vectors = prepared["embeddings"][prepared["article_index"][positions]]
        local_order = _greedy_mmr_order(relevance, vectors, prepared["k"], weight)
        result[positions[local_order]] = np.arange(1, len(positions) + 1)
        if progress_every and completed % progress_every == 0:
            print(f"MMR lambda={weight:g}: {completed:,}/{len(eligible):,} eligible slates processed.")
    if progress_every:
        print(f"MMR lambda={weight:g}: complete; {len(eligible):,} eligible slates processed.")
    return result

In [ ]:
# --- 1.4.3b: Expose a DataFrame adapter for the existing relevance/diversity helpers. ---
def apply_mmr(prepared, lambda_value, scaling=None, progress_every=0):
    """Return one lambda's full result; keep original score and baseline rank columns."""
    final_ranks = run_mmr_ranks(prepared, lambda_value, scaling, progress_every)
    result = prepared["reference"].copy()
    result["rerank_rank"] = final_ranks
    result["rerank_score"] = -final_ranks

    # Keep experiment settings alongside the in-memory result; exports come later.
    result.attrs["mmr_settings"] = {
        "lambda": float(lambda_value),
        "relevance_scaling": scaling if float(lambda_value) < 1 else "identity (not applied)",
        "top_k": prepared["k"],
        "first_selection": "baseline rank 1",
        "tie_rule": "lowest baseline rank within the numerical objective tie band",
        "cosine_tie_atol": MMR_COSINE_TIE_ATOL,
        "objective_tie_atol": (1.0 - float(lambda_value)) * MMR_COSINE_TIE_ATOL,
        "similarity": "maximum signed cosine to already-selected articles",
        "tail_order": "baseline order of unselected candidates",
    }
    return result

#### 1.4.4 Prepare tuning arrays and check lambda = 1

Lambda = 1 must reproduce every baseline rank, including tied scores. This tests the ranking implementation, not upstream cleaning.

In [ ]:
# --- 1.4.4: Prepare tuning arrays and verify the relevance-only identity. ---

# This assertion checks the algorithm identity rather than revalidating source rows.
mmr_prepared = prepare_mmr_inputs(baseline_ranked, emb_lookup, k=TOP_K)
np.testing.assert_array_equal(run_mmr_ranks(mmr_prepared, 1.0, RELEVANCE_SCALING),
                              baseline_ranked["baseline_rank"].to_numpy())
print("Lambda = 1 preserves the baseline order exactly.")

### 1.5 Small correctness checks with synthetic inputs

Check the MMR implementation with tiny invented slates, deterministic invariants, and an independent greedy reference. The examples are labelled **SYNTHETIC**: their metric changes test software behavior and do not establish model quality or an optimal lambda for MIND.

Toy inputs, results, and helpers use `synthetic_` / `_synthetic_` names. Their local seeds, scaling choices, lambda values, and k values leave the actual experiment settings and data unchanged. These checks execute after the preceding preparation cells in the complete ordered run; an assertion failure stops execution with its description.

#### 1.5.1 A hand-worked example: promoting a dissimilar candidate

Five invented articles have descending relevance scores. A, B, and C are similar news stories; D is an orthogonal sports story outside the original top 3. With the synthetic choice lambda = 0.5 and unscaled scores, A is selected first. B's next objective is `0.5 * 0.90 - 0.5 * 1 = -0.05`, while D's is `0.5 * 0.80 - 0.5 * 0 = 0.40`, so D moves ahead. The expected selected order is **A, D, B**, followed by C and E in baseline order.

D is marked cold only to make the exposure calculation visible; that flag does not influence MMR. Print the candidate ranks and before/after top lists, followed by the same relevance/diversity summaries used for real data. The example illustrates that increased diversity can cost click-based relevance and that cold share can change without a cold-item bonus.

In [ ]:
# --- 1.5.1a: Build a tiny, clearly labelled synthetic demonstration. ---
def _synthetic_make_reference(impression_id, scores, k, seed=42):
    """Create invented candidates and reuse the real baseline-ranking helper."""
    size = len(scores)
    frame = pd.DataFrame({
        "impression_id": [impression_id] * size,
        "news_id": [f"{impression_id}_{i:03d}" for i in range(size)],
        "label": [1] + [0] * (size - 1),
        COLD_COLUMN: [0] * size,
        "category": ["synthetic_news"] * size,
        "score": scores,
    })
    return build_relevance_reference(frame, k=k, seed=seed)[0]


def _synthetic_build_demo():
    """Return toy-only objects; never write into tuning/evaluation experiment state."""
    k = 3
    frame = pd.DataFrame({
        "impression_id": ["synthetic_demo"] * 5,
        "news_id": ["toy_A", "toy_B", "toy_C", "toy_D", "toy_E"],
        "score": [0.95, 0.90, 0.85, 0.80, 0.10],
        "label": [1, 0, 1, 0, 0],
        COLD_COLUMN: [0, 0, 0, 1, 1],
        "category": ["synthetic_news", "synthetic_news", "synthetic_news",
                     "synthetic_sports", "synthetic_news"],
    })
    reference, _, baseline_top = build_relevance_reference(frame, k=k, seed=42)
    lookup = {
        "toy_A": np.array([1.0, 0.0]), "toy_B": np.array([1.0, 0.0]),
        "toy_C": np.array([1.0, 0.0]), "toy_D": np.array([0.0, 1.0]),
        "toy_E": np.array([1.0, 0.0]),
    }
    prepared = prepare_mmr_inputs(reference, lookup, k=k)
    result = apply_mmr(prepared, lambda_value=0.5, scaling="none")
    result_top = topk(result.loc[result["rerank_rank"].le(k)], "rerank_rank", k=k)
    expected = ["toy_A", "toy_D", "toy_B"]
    if result_top["news_id"].tolist() != expected:
        raise AssertionError("SYNTHETIC demo: expected top-3 order A, D, B.")
    return {"k": k, "reference": reference, "baseline_top": baseline_top,
            "lookup": lookup, "prepared": prepared, "result": result, "result_top": result_top}

In [ ]:
# --- 1.5.1b: Inspect the toy ranks and verify their observable metric changes. ---
synthetic_demo = _synthetic_build_demo()
print("SYNTHETIC EXAMPLE ONLY: lambda=0.5, scaling='none', k=3.")
print(synthetic_demo["result"][["news_id", "score", "label", COLD_COLUMN, "category",
                                "baseline_rank", "rerank_rank"]].to_string(index=False))
print("Before top-3:", synthetic_demo["baseline_top"]["news_id"].tolist())
print("After top-3: ", synthetic_demo["result_top"]["news_id"].tolist())

# Reuse the actual summaries; MRR/nDCG keep their configured @5/@10 cutoffs.
synthetic_demo_relevance = pd.concat({
    "before": summarize_reference_relevance(synthetic_demo["reference"]),
    "after": summarize_reference_relevance(synthetic_demo["result"], rank_col="rerank_rank"),
}, names=["ranking"])
synthetic_demo_diversity = pd.concat({
    "before": summarize_reference_diversity(synthetic_demo["baseline_top"], synthetic_demo["lookup"], k=3),
    "after": summarize_reference_diversity(synthetic_demo["result_top"], synthetic_demo["lookup"], k=3),
}, names=["ranking"])

# Expected values are hand-calculated for this one invented impression.
assert np.isclose(synthetic_demo_diversity.loc[("before", "eligible_impressions"), "unique_categories"], 1)
assert np.isclose(synthetic_demo_diversity.loc[("after", "eligible_impressions"), "unique_categories"], 2)
assert np.isclose(synthetic_demo_diversity.loc[("before", "eligible_impressions"), "intra_list_sim"], 1)
assert np.isclose(synthetic_demo_diversity.loc[("after", "eligible_impressions"), "intra_list_sim"], 1/3)
assert np.isclose(synthetic_demo_diversity.loc[("before", "eligible_impressions"), "cold_share"], 0)
assert np.isclose(synthetic_demo_diversity.loc[("after", "eligible_impressions"), "cold_share"], 1/3)
print("SYNTHETIC relevance (AUC stays tied to original model scores):")
print(synthetic_demo_relevance.to_string(float_format=lambda value: f"{value:.4f}"))
print("SYNTHETIC diversity/exposure (these are not MIND results):")
print(synthetic_demo_diversity.to_string(float_format=lambda value: f"{value:.4f}"))

#### 1.5.2 Deterministic MMR invariants

Check full permutations, preserved scores/first article, short/exact-k slates, deterministic ties, signed cosine, lambda = 1, duplicate-vector rounding, and objective gaps inside/outside the numerical tie band on invented slates. These test our ranking implementation, not real-data cleaning.

In [ ]:
# --- 1.5.2: Check valid-input MMR behavior on small synthetic slates. ---
def run_synthetic_invariant_checks():
    """Exercise ranking behavior without modifying real experiment variables."""
    checks = []
    for size in [2, 3, 6]:
        reference = _synthetic_make_reference(f"synthetic_{size}", np.ones(size), k=3)
        lookup = {news_id: np.array([1., 0.]) for news_id in reference["news_id"]}
        prepared = prepare_mmr_inputs(reference, lookup, k=3)
        for weight in [0., .5, 1.]:
            result = apply_mmr(prepared, weight, "per_impression_minmax")
            # Every article occurs once; original scores and the first item stay intact.
            np.testing.assert_array_equal(np.sort(result["rerank_rank"]), np.arange(1, size + 1))
            pd.testing.assert_series_equal(result["score"], reference["score"])
            assert result.loc[result["baseline_rank"].eq(1), "rerank_rank"].item() == 1
            # Identical scores/vectors resolve by baseline rank at every lambda.
            np.testing.assert_array_equal(result["rerank_rank"], reference["baseline_rank"])
            checks.append({"case": "constant_scores_vectors", "slate_size": size, "lambda": weight, "result": "PASS"})
    # Opposite vectors retain signed cosine -1 rather than an artificial zero floor.
    order = _greedy_mmr_order(np.array([1., .9, .8]),
                              np.array([[1., 0.], [1., 0.], [-1., 0.]]), 2, .5)
    np.testing.assert_array_equal(order, [0, 2, 1])
    checks.append({"case": "signed_cosine", "slate_size": 3, "lambda": .5, "result": "PASS"})

    # Regression: duplicate vectors at different matrix-row positions can round differently.
    # Before the tie fix, this seeded fixture selected position 4 instead of position 1.
    raw_vectors = np.random.default_rng(6).normal(size=(2, 7))
    unit_vectors = (raw_vectors / np.linalg.norm(raw_vectors, axis=1, keepdims=True)).astype(np.float32)
    duplicated = unit_vectors[[0, 1, 0, 0, 1]]
    for weight in [0., .5, .9]:
        observed = _greedy_mmr_order(np.zeros(5), duplicated, 2, weight)
        np.testing.assert_array_equal(observed, [0, 1, 2, 3, 4])
        checks.append({"case": "duplicate_vector_roundoff", "slate_size": 5, "lambda": weight, "result": "PASS"})

    # The tolerance is narrow: a gap beyond the band still wins on its actual objective.
    for cosine_gap, expected_winner, name in [
        (0.5 * MMR_COSINE_TIE_ATOL, 1, "inside_tie_band"),
        (2.0 * MMR_COSINE_TIE_ATOL, 2, "outside_tie_band"),
    ]:
        vectors = np.array([[1., 0.], [0., 1.],
                            [-cosine_gap, np.sqrt(1. - cosine_gap ** 2)]], dtype=np.float32)
        observed = _greedy_mmr_order(np.zeros(3), vectors, 2, .5)
        assert observed[1] == expected_winner
        checks.append({"case": name, "slate_size": 3, "lambda": .5, "result": "PASS"})
    return pd.DataFrame(checks)


synthetic_invariant_results = run_synthetic_invariant_checks()
print(synthetic_invariant_results.to_string(index=False))

#### 1.5.3 Compare with a simple independent greedy reference

For small seeded random slates, compare the incremental implementation with a deliberately simple reference that recomputes similarities to **every selected article** at each step. The reference uses its own direct min-max calculation on ordinary finite toy scores and the same declared numerical tie convention. This checks that maintaining the maximum similarity incrementally produces the same order as the formula, including lambda = 0 and lambda = 1.

Use 20 small slates, five lambda settings, and both scaling methods: **200 comparisons**. This is a correctness check, not a hyperparameter search or speed benchmark. The temporary seed, lambda list, scaling choices, and k values are local to this check and do not change project settings.

In [ ]:
# --- 1.5.3a: Define an independent, intentionally simple greedy reference and comparison. ---
def _synthetic_simple_mmr_order(scores, vectors, k, lambda_value, scaling):
    """Recompute all selected similarities; used only for small correctness cases."""
    if lambda_value == 1.0:
        return np.arange(len(scores))
    if scaling == "none":
        relevance = np.array(scores, dtype=np.float64)
    else:
        spread = np.max(scores) - np.min(scores)
        relevance = np.zeros(len(scores)) if spread == 0 else (scores - np.min(scores)) / spread
    selected = [0]
    while len(selected) < k:
        remaining = [i for i in range(len(scores)) if i not in selected]
        objectives = []
        for candidate in remaining:
            penalty = max(float(np.clip(np.dot(vectors[candidate], vectors[previous]), -1., 1.))
                          for previous in selected)
            objectives.append(lambda_value * relevance[candidate] - (1 - lambda_value) * penalty)
        # Apply the same declared numerical convention with plain Python selection.
        best = max(objectives)
        tie_band = (1.0 - lambda_value) * MMR_COSINE_TIE_ATOL
        selected.append(next(candidate for candidate, objective in zip(remaining, objectives)
                             if objective >= best - tie_band))
    return np.array(selected + [i for i in range(len(scores)) if i not in selected])


def _synthetic_check_against_reference():
    """Compare 200 toy runs without retaining their candidate-level result tables."""
    rng = np.random.default_rng(2026)
    comparisons = 0
    for trial in range(20):
        n = int(rng.integers(4, 20))
        k = int(rng.integers(1, n))
        scores = np.sort(rng.normal(size=n))[::-1]
        reference = _synthetic_make_reference(f"synthetic_random_{trial}", scores, k=k)
        lookup = dict(zip(reference["news_id"], rng.normal(size=(n, 8))))
        prepared = prepare_mmr_inputs(reference, lookup, k=k)
        positions = prepared["order"]
        ordered_scores = prepared["scores"][positions]
        vectors = prepared["embeddings"][prepared["article_index"][positions]]
        for weight in [0.0, 0.1, 0.5, 0.9, 1.0]:
            for scaling in ["none", "per_impression_minmax"]:
                expected = _synthetic_simple_mmr_order(ordered_scores, vectors, k, weight, scaling)
                ranks = run_mmr_ranks(prepared, weight, scaling)
                observed = np.argsort(ranks[positions])
                if not np.array_equal(observed, expected):
                    raise AssertionError(
                        f"SYNTHETIC reference mismatch: trial={trial}, lambda={weight}, scaling={scaling}"
                    )
                comparisons += 1
    return comparisons

#### 1.5.4 Summarize software correctness

Compare the optimized MMR with the independent greedy reference over 200 synthetic cases. These use their own seed and never select a real-data lambda.

In [ ]:
# --- 1.5.4: Run the independent greedy-reference comparisons. ---

# The independent implementation recomputes similarities for each selection.
synthetic_reference_comparisons = _synthetic_check_against_reference()
assert synthetic_reference_comparisons == 200
print(f"Passed {len(synthetic_invariant_results)} deterministic cases and 200 independent comparisons.")

## 2. Lambda sweep and selection on the tuning split

Record the tuning specification, sweep the shared lambda grid, and compare each setting with the relevance-only baseline. Apply the declared relevance-loss budget and diversity objective using tuning data, then retain the selected settings and evidence for held-out evaluation. Keep compact summaries and release each temporary candidate-level ranking after measuring it.

### 2.1 Record tuning experiment settings

Record the shared seed, top-k, model/encoder identifiers, scaling, input paths, and tie/similarity conventions. Build a unique descending lambda grid that includes lambda = 1 as the relevance-only reference. These settings accompany the tuning summaries and are copied into the evaluation record in 2.5-3.1.

In [ ]:
# --- 2.1: Record the specification used by this tuning run. ---
tuning_settings = {
    "model": BASE_MODEL_NAME, "embedding_encoder_id": EMBEDDING_ENCODER_ID,
    "split": "tune", "tune_date": str(config.TUNE_DATE), "top_k": TOP_K, "seed": SEED,
    "lambdas": tuple(sorted({1.0, *(validate_mmr_lambda(v) for v in LAMBDA_VALUES)}, reverse=True)),
    "relevance_scaling": RELEVANCE_SCALING, "cold_column": COLD_COLUMN, "score_column": SCORE_COLUMN,
    "candidate_path": str(TUNE_CANDIDATES_PATH), "score_path": str(TUNE_SCORES_PATH),
    "embedding_path": str(NEWS_EMBEDDINGS_PATH), "embedding_ids_path": str(NEWS_EMBEDDING_IDS_PATH),
    "eligibility": "full slate size > top_k",
    "baseline_ties": "shared seeded ties after canonical candidate-key sorting",
    "mmr_ties": "lowest baseline rank within (1 - lambda) * mmr_cosine_tie_atol of the best objective",
    "mmr_cosine_tie_atol": MMR_COSINE_TIE_ATOL, "similarity": "maximum signed cosine",
}
# Scaling is a required experiment choice, rather than a guessed default.
if RELEVANCE_SCALING not in SUPPORTED_RELEVANCE_SCALING:
    raise ValueError(f"Set RELEVANCE_SCALING to one of {SUPPORTED_RELEVANCE_SCALING} in 1.1.")

### 2.2 Run the tuning lambda sweep

Reuse the prepared candidate arrays and article-vector context across lambdas. Measure each completed ranking, retain compact full-precision summaries, and release its temporary candidate-level result before the next iteration. Lambda = 1 reuses the reference summaries.

#### 2.2.1 Prepare reusable metric inputs

Gather a compact article matrix. Original model-score AUC is invariant under reranking, so reuse its baseline value and compute the explicit-order relevance metrics for each final ranking.

In [ ]:
# --- 2.2.1: Prepare the invariant article matrix and summarize explicit rank relevance. ---
def prepare_sweep_metric_context(reference, lookup, settings):
    """Cache only active vectors, normalized exactly as utils.metrics.topk_diversity."""
    ids = pd.Index(reference["news_id"].unique())
    matrix = np.stack([lookup[news_id] for news_id in ids]).astype(np.float32)
    matrix /= np.maximum(np.linalg.norm(matrix, axis=1, keepdims=True), 1e-12)
    return {"news_ids": ids, "vectors": matrix, "k": settings["top_k"],
            "seed": settings["seed"], "cold_column": settings["cold_column"]}


def summarize_sweep_relevance(ranked, rank_column, baseline_relevance, context):
    """Use shared ranking/segment metrics while reusing invariant original-score AUC."""
    tables = []
    for scope in ["all_impressions", "eligible_impressions"]:
        selected = ranked if scope == "all_impressions" else ranked.loc[ranked["rerank_eligible"]]
        if selected.empty:
            summary = baseline_relevance.loc[scope].copy()
        else:
            inputs = selected[["impression_id", "label", context["cold_column"]]].copy()
            inputs["_order_score"] = -selected[rank_column].to_numpy(dtype=np.float64)
            summary = p6_evaluate_by_segment_raw(inputs, "_order_score", context["cold_column"],
                                          seed=context["seed"]).drop(columns="auc")
            summary.insert(0, "model_score_auc", baseline_relevance.loc[scope, "model_score_auc"])
        summary.index.name = "segment"
        tables.append(summary)
    return pd.concat(tables, keys=["all_impressions", "eligible_impressions"], names=["scope"])

#### 2.2.2 Define full-list diversity metrics

Calculate category count, signed pairwise cosine, and cold share on full top-k lists. Exclude short lists consistently and report their counts. Empty scopes and k = 1 produce unavailable semantic values rather than fabricated zeros.

In [ ]:
# --- 2.2.2: Compute full-list diversity with reusable vectors. ---
def summarize_sweep_diversity(top, context, semantic_batch_size=2048):
    """Match Section 1 diversity metrics without repeatedly stacking the lookup."""
    k = context["k"]
    rows = {}
    for scope in ["all_full_topk_lists", "eligible_impressions"]:
        selected = top if scope == "all_full_topk_lists" else top.loc[top["rerank_eligible"]]
        sizes = selected.groupby("impression_id", sort=False).size()
        if not sizes.eq(k).any():
            result = {"unique_categories": np.nan, "intra_list_sim": np.nan, "cold_share": np.nan,
                      "n_lists": 0, "n_short_lists_excluded": len(sizes)}
            status = "NO_FULL_LISTS"
        else:
            # Let the shared helper define category/cold-share means and exclusions.
            result = p6_topk_diversity_raw(selected, category_col="category", emb_lookup=None,
                                    cold_col=context["cold_column"], k=k)
            full = selected.loc[selected["impression_id"].isin(sizes.index[sizes.eq(k)])]
            full = full.sort_values("impression_id", kind="stable")
            indices = context["news_ids"].get_indexer(full["news_id"])
            if k == 1:
                result["intra_list_sim"] = np.nan
                status = "UNDEFINED_SINGLE_ITEM_LIST"
            else:
                indices = indices.reshape(-1, k)
                similarity = np.empty(len(indices), dtype=np.float64)
                triangle = np.triu_indices(k, 1)
                for start in range(0, len(indices), semantic_batch_size):
                    vectors = context["vectors"][indices[start:start + semantic_batch_size]]
                    cosine = np.clip(np.einsum("nkd,njd->nkj", vectors, vectors), -1.0, 1.0)
                    similarity[start:start + semantic_batch_size] = cosine[:, triangle[0], triangle[1]].mean(axis=1, dtype=np.float64)
                result["intra_list_sim"] = float(similarity.mean())
                status = "AVAILABLE"
        result["n_input_impressions"] = len(sizes)
        result["semantic_status"] = status
        rows[scope] = result
    table = pd.DataFrame.from_dict(rows, orient="index")
    table.index.name = "scope"
    return table[["unique_categories", "intra_list_sim", "cold_share", "n_lists",
                  "n_short_lists_excluded", "n_input_impressions", "semantic_status"]]

#### 2.2.3 Execute the full sweep

Run one lambda at a time. Lambda = 1 reuses baseline metrics; other lambdas use the same scaling and tie rules. Keep unrounded summaries for selection.

In [ ]:
# --- 2.2.3: Run the tuning sweep directly, retaining compact metric summaries. ---
metric_context = prepare_sweep_metric_context(baseline_ranked, emb_lookup, tuning_settings)
tuning_sweep_summaries = {}
for weight in tuning_settings["lambdas"]:
    if weight == 1.0:
        # The reference summaries already describe this exact order.
        relevance = baseline_relevance.copy()
        diversity = baseline_diversity.copy()
    else:
        ranked = apply_mmr(mmr_prepared, weight, RELEVANCE_SCALING, MMR_PROGRESS_EVERY)
        relevance = summarize_sweep_relevance(ranked, "rerank_rank", baseline_relevance, metric_context)
        ranked_top = topk(ranked.loc[ranked["rerank_rank"].le(TOP_K)], "rerank_rank", k=TOP_K)
        diversity = summarize_sweep_diversity(ranked_top, metric_context)
        del ranked, ranked_top  # Retain summaries, not a full candidate table per lambda.
    tuning_sweep_summaries[weight] = {"relevance": relevance, "diversity": diversity}
    print(f"Completed tuning metrics for lambda={weight:g}.")
del metric_context

### 2.3 Assemble and inspect tuning comparisons

Combine the completed sweep summaries and calculate signed changes against lambda = 1. Preserve metric scopes, segments, population counts, and unavailable-value labels. Round printed views only; lambda selection uses the stored full-precision values.

#### 2.3.1 Define raw comparison tables

Preserve scopes/segments and population counts. Relative change divides by the absolute reference magnitude, keeping the sign meaningful for signed cosine.

In [ ]:
# --- 2.3.1: Define explicit missing/zero-reference handling for signed metric changes. ---
def _tuning_metric_change(value, reference):
    """Return signed delta, relative percent, and an explicit availability status."""
    if pd.isna(reference):
        return np.nan, np.nan, "MISSING_BASELINE"
    if pd.isna(value):
        return np.nan, np.nan, "MISSING_VALUE"
    delta = float(value) - float(reference)
    if reference == 0:
        return delta, np.nan, "ZERO_BASELINE"
    return delta, 100.0 * delta / abs(float(reference)), "AVAILABLE"


def build_tuning_comparison_tables(summaries, settings):
    """Assemble compact summaries without rerunning ranking or data loading."""
    grid = settings["lambdas"]
    relevance = pd.concat([summaries[w]["relevance"] for w in grid], keys=grid, names=["lambda"])
    diversity = pd.concat([summaries[w]["diversity"] for w in grid], keys=grid, names=["lambda"])
    rel_metrics = ["model_score_auc", "mrr", "ndcg@5", "ndcg@10"]
    div_metrics = ["unique_categories", "intra_list_sim", "cold_share"]
    rows = []
    for weight in grid:
        record = summaries[weight]
        row = {"lambda": weight, "model": settings["model"], "split": settings["split"],
               "top_k": settings["top_k"], "seed": settings["seed"],
               "relevance_scaling": settings["relevance_scaling"]}
        for (scope, segment), values in record["relevance"].iterrows():
            for metric in rel_metrics + ["n_impressions", "n_skipped"]:
                key = f"relevance__{scope}__{segment}__{metric}"
                row[key] = values[metric]
                if metric in rel_metrics:
                    baseline_value = summaries[1.0]["relevance"].loc[(scope, segment), metric]
                    delta, relative, status = _tuning_metric_change(values[metric], baseline_value)
                    row[key + "__delta"] = delta
                    row[key + "__relative_change_pct"] = relative
                    row[key + "__change_status"] = status
        for scope, values in record["diversity"].iterrows():
            for metric in div_metrics + ["n_lists", "n_short_lists_excluded", "n_input_impressions", "semantic_status"]:
                key = f"diversity__{scope}__{metric}"
                row[key] = values[metric]
                if metric in div_metrics:
                    baseline_value = summaries[1.0]["diversity"].loc[scope, metric]
                    delta, relative, status = _tuning_metric_change(values[metric], baseline_value)
                    row[key + "__delta"] = delta
                    row[key + "__relative_change_pct"] = relative
                    row[key + "__change_status"] = status
        rows.append(row)
    wide = pd.DataFrame(rows).set_index("lambda")

    # Retain counts as integers even though iterrows mixes them with floating metrics.
    count_columns = [c for c in wide.columns if c.endswith(("__n_impressions", "__n_skipped", "__n_lists",
                                                            "__n_short_lists_excluded", "__n_input_impressions"))]
    wide[count_columns] = wide[count_columns].astype(np.int64)
    wide.attrs["experiment_settings"] = deepcopy(settings)
    return wide, relevance, diversity

#### 2.3.2 Build and inspect comparisons

Build `tuning_results` and the relevance/diversity comparison tables from the completed sweep. `tuning_results` retains each metric, signed delta, relative change, availability label, and population count for export. Inspect formatted relevance and diversity views while retaining the unrounded tables for selection, plots, and exports.

In [ ]:
# --- 2.3.2: Build the unrounded comparison tables and print formatted views. ---
(tuning_results, tuning_relevance_comparison, tuning_diversity_comparison) = build_tuning_comparison_tables(tuning_sweep_summaries, tuning_settings)
# Printing changes only the display precision, not stored values.
print("Relevance by scope and clicked-article segment:")
print(tuning_relevance_comparison.to_string(float_format=lambda value: f"{value:.4f}"))
print("Diversity, cold exposure, and list populations:")
print(tuning_diversity_comparison.to_string(float_format=lambda value: f"{value:.4f}"))

### 2.4 Apply an explicit tuning selection policy

Assess each completed lambda against a **declared relevance-loss budget**, then rank qualifying settings by the chosen diversity objective. Print both eligibility and reasons so the decision is reviewable. The agreed policy uses eligible-impression nDCG@10, minimizes intra-list similarity, and has no optional guards. Compare candidate loss budgets in 2.4.4, then explicitly choose the final budget.

Outputs are `tuning_selection_table`, a validated policy/settings record, and `proposed_tuning_lambda` when a setting qualifies. **`SELECTED_LAMBDA` and `selected_tuning_settings` remain unset** until the settings handoff in 2.5. A proposed candidate is a tuning result, not evidence of held-out improvement. If nothing qualifies, report that outcome without falling back automatically to lambda = 1.

Declare the policy before inspecting outcomes where possible. If it is revised after reviewing tuning results, document the revision in `SELECTION_POLICY_NOTE`. Never use the held-out evaluation set to choose or revise this rule.

#### 2.4.1 Declare the selection policy

The configured policy uses `ndcg@10`, `intra_list_sim`, and `eligible_impressions`. P2's current diagnostics notebook specifies no primary selection metric; the shared evaluator and other models report nDCG@10, so it is used here for top-10 relevance. P5 selects its NRMS checkpoint by tune AUC; this separate reranking policy measures changes in top-10 order. Update the single relevance-metric variable if P2/the group agrees another metric.

The remaining decision is `MAX_RELATIVE_RELEVANCE_LOSS`. Compare `RELEVANCE_LOSS_BUDGETS` in 2.4.4 using tuning data, then set the chosen fraction before the complete run.

| Variable | Meaning |
|---|---|
| `SELECTION_RELEVANCE_METRIC` | `"mrr"`, `"ndcg@5"`, or `"ndcg@10"`. |
| `RELEVANCE_LOSS_BUDGETS` | `(0.0, 0.005, 0.01, 0.02, 0.05)` for tuning-only budget comparison. |
| `MAX_RELATIVE_RELEVANCE_LOSS` | Allowed loss fraction relative to lambda = 1; `0.01` means 1%. |
| `SELECTION_DIVERSITY_METRIC` | `"intra_list_sim"` to minimize signed similarity, or `"unique_categories"` to maximize variety. |
| `SELECTION_SCOPE` | `"overall"` or `"eligible_impressions"`; overall relevance and diversity retain their respective population definitions from 1.3. |
| `SELECTION_CONSTRAINTS` | `{}` or optional `min_cold_share_delta` and/or `max_segment_relative_relevance_loss` constraints. Cold-share change is a signed fraction, so `0.02` means +2 percentage points. Segment limits map cold/warm/mixed names to relative loss fractions. |
| `SELECTION_POLICY_NOTE` | Optional rationale, including any revisions after inspecting tuning results. |

The primary budget and optional segment budgets use `(reference - value) / reference`. Relative relevance-loss calculations require an available positive reference; the cold-share guard uses an absolute change. Held-out outcomes are not used to revise this policy.

`SELECTION_CONSTRAINTS = {}` disables extra rules. They are optional policy requirements, such as requiring cold exposure not to decrease or limiting the relevance loss within a cold/warm/mixed segment. They are not data checks and are not needed to run MMR. The primary relevance-loss budget still applies; cold exposure and segment results remain reported without additional guards.

In [ ]:
# --- 2.4.1b: Configure the agreed policy and compare candidate relevance-loss budgets. ---
# The metric/scope/objective are agreed; choose the final loss budget from tuning evidence.
SELECTION_RELEVANCE_METRIC = "ndcg@10"  # Shared top-10 relevance metric; update if P2 agrees another primary metric.
RELEVANCE_LOSS_BUDGETS = (0.0, 0.005, 0.01, 0.02, 0.05)  # Tuning sensitivity analysis.
MAX_RELATIVE_RELEVANCE_LOSS = None      # Choose after reviewing 2.4.4; 0.01 means 1% loss.
SELECTION_DIVERSITY_METRIC = "intra_list_sim"  # Minimize mean pairwise signed cosine.
SELECTION_SCOPE = "eligible_impressions"      # Select using the impressions MMR can change.
SELECTION_CONSTRAINTS = {}             # No extra exposure/segment guards for now.
SELECTION_POLICY_NOTE = ""             # Optional rationale/revision note for the final record.

# These policy choices are new here; later tables show their effects on lambda eligibility.
print(pd.Series({
    "relevance_metric": SELECTION_RELEVANCE_METRIC,
    "max_relative_relevance_loss": MAX_RELATIVE_RELEVANCE_LOSS,
    "diversity_metric": SELECTION_DIVERSITY_METRIC,
    "scope": SELECTION_SCOPE,
    "constraints": SELECTION_CONSTRAINTS,
}).to_string())

#### 2.4.2 Interpret policy settings

Validate the experiment's policy choices, not the processed data. Keep budget/objective definitions explicit, including whether eligible slates or overall results drive selection.

In [ ]:
# --- 2.4.2: Interpret the declared tuning policy. ---

# Validate policy choices here; candidate cleaning is owned by preprocessing.
def _selection_finite_number(value, name, lower, upper):
    """Validate policy numbers without accepting bools, strings, NaN, or infinity."""
    if (isinstance(value, (bool, np.bool_))
            or not isinstance(value, (int, float, np.integer, np.floating))
            or not np.isfinite(value) or not lower <= value <= upper):
        raise ValueError(f"{name} must be a finite number in [{lower}, {upper}].")
    return float(value)


def read_tuning_selection_policy(state):
    """Return a normalized policy, or missing field names without inventing choices."""
    required = ["SELECTION_RELEVANCE_METRIC", "MAX_RELATIVE_RELEVANCE_LOSS",
                "SELECTION_DIVERSITY_METRIC", "SELECTION_SCOPE", "SELECTION_CONSTRAINTS"]
    missing = [name for name in required if state.get(name) is None]
    if missing:
        return None, missing
    metric = state["SELECTION_RELEVANCE_METRIC"]
    objective = state["SELECTION_DIVERSITY_METRIC"]
    scope = state["SELECTION_SCOPE"]
    if not isinstance(metric, str) or metric not in ("mrr", "ndcg@5", "ndcg@10"):
        raise ValueError("Selection relevance metric must be mrr, ndcg@5, or ndcg@10; model-score AUC is invariant.")
    if not isinstance(objective, str) or objective not in ("intra_list_sim", "unique_categories"):
        raise ValueError("Selection diversity metric must be intra_list_sim or unique_categories.")
    if not isinstance(scope, str) or scope not in ("eligible_impressions", "overall"):
        raise ValueError("Selection scope must be eligible_impressions or overall.")
    budget = _selection_finite_number(state["MAX_RELATIVE_RELEVANCE_LOSS"], "Relative relevance-loss budget", 0, 1)
    supplied = state["SELECTION_CONSTRAINTS"]
    if not isinstance(supplied, dict):
        raise ValueError("SELECTION_CONSTRAINTS must be a dictionary; use {} to disable optional guards.")
    unknown = set(supplied) - {"min_cold_share_delta", "max_segment_relative_relevance_loss"}
    if unknown:
        raise ValueError(f"Unknown selection constraints: {sorted(map(str, unknown))}")
    constraints = {}
    if "min_cold_share_delta" in supplied:
        constraints["min_cold_share_delta"] = _selection_finite_number(
            supplied["min_cold_share_delta"], "Minimum cold-share delta", -1, 1)
    if "max_segment_relative_relevance_loss" in supplied:
        segment_limits = supplied["max_segment_relative_relevance_loss"]
        if not isinstance(segment_limits, dict) or set(segment_limits) - {"cold", "warm", "mixed"}:
            raise ValueError("Segment limits must be a dictionary with cold/warm/mixed keys only.")
        constraints["max_segment_relative_relevance_loss"] = {
            segment: _selection_finite_number(value, f"{segment} relative-loss budget", 0, 1)
            for segment, value in segment_limits.items()
        }
    note = state.get("SELECTION_POLICY_NOTE", "")
    if not isinstance(note, str):
        raise ValueError("SELECTION_POLICY_NOTE must be a string.")
    return {
        "relevance_metric": metric, "max_relative_relevance_loss": budget,
        "diversity_metric": objective, "scope": scope,
        "relevance_scope": "eligible_impressions" if scope == "eligible_impressions" else "all_impressions",
        "diversity_scope": "eligible_impressions" if scope == "eligible_impressions" else "all_full_topk_lists",
        "objective_ascending": objective == "intra_list_sim", "constraints": constraints,
        "tie_break": ("primary_relevance_desc", "lambda_desc"),
        "boundary_tolerance": 1e-12, "note": note,
    }, []

#### 2.4.3 Explain eligibility and propose the best qualifying setting

For each lambda, show primary relevance and its reference/loss, diversity, cold-share change, reporting counts, segment diagnostics, and the reason for qualifying or failing. Evaluate only explicitly requested optional guards. Among qualifying settings, minimize similarity or maximize category variety, then prefer higher primary relevance and larger lambda. Lambda = 1 is eligible under the same rules and may win; diversification is not forced when the policy does not support it.

`selection_rank` applies only to qualifying settings. A missing/zero reference for any requested guard produces an explicit failure reason. With no qualifiers, print the full eligibility/reason table, then stop with an ordinary `ValueError`. No fallback lambda is selected.

In [ ]:
# --- 2.4.3: Evaluate the declared constraints and order only qualifying settings. ---
def _selection_relative_loss(value, reference):
    """Return a loss fraction only when the reference supports that calculation."""
    if not np.isfinite(value) or not np.isfinite(reference) or reference <= 0:
        return np.nan
    return (float(reference) - float(value)) / float(reference)


def _selection_at_least(value, minimum, tolerance):
    """Handle floating arithmetic at a boundary without rounding away real differences."""
    return bool(np.isfinite(value) and (value >= minimum or abs(value - minimum) <= tolerance))


def propose_tuning_lambda(relevance, diversity, settings, policy):
    """Return a reviewable eligibility table and a proposal; do not mutate experiment state."""
    metric = policy["relevance_metric"]
    rel_scope, div_scope = policy["relevance_scope"], policy["diversity_scope"]
    objective = policy["diversity_metric"]
    base_relevance = float(relevance.loc[(1.0, rel_scope, "all"), metric])
    base_diversity = float(diversity.loc[(1.0, div_scope), objective])
    base_cold = float(diversity.loc[(1.0, div_scope), "cold_share"])
    budget = policy["max_relative_relevance_loss"]
    tolerance = policy["boundary_tolerance"]
    segment_limits = policy["constraints"].get("max_segment_relative_relevance_loss", {})
    rows = []
    for weight in settings["lambdas"]:
        r = relevance.loc[(weight, rel_scope, "all")]
        d = diversity.loc[(weight, div_scope)]
        value = float(r[metric])
        loss = _selection_relative_loss(value, base_relevance)
        failures = []
        if not np.isfinite(base_relevance) or base_relevance <= 0:
            failures.append("PRIMARY_BASELINE_MISSING_OR_NONPOSITIVE")
        elif not np.isfinite(value):
            failures.append("PRIMARY_VALUE_UNAVAILABLE")
        elif not _selection_at_least(value, base_relevance * (1 - budget), tolerance):
            failures.append("PRIMARY_RELEVANCE_LOSS_EXCEEDS_BUDGET")
        diversity_value = float(d[objective])
        if not np.isfinite(diversity_value):
            failures.append("DIVERSITY_OBJECTIVE_UNAVAILABLE")
        cold_delta = float(d["cold_share"]) - base_cold
        if "min_cold_share_delta" in policy["constraints"]:
            if not _selection_at_least(cold_delta, policy["constraints"]["min_cold_share_delta"], tolerance):
                failures.append("COLD_SHARE_GUARD_FAILED_OR_UNAVAILABLE")
        row = {
            "lambda": weight, "primary_relevance": value, "reference_relevance": base_relevance,
            "relevance_delta": value - base_relevance, "relative_relevance_loss": loss,
            "relative_loss_budget": budget, "diversity_value": diversity_value,
            "reference_diversity": base_diversity, "diversity_delta": diversity_value - base_diversity,
            "unique_categories": d["unique_categories"], "intra_list_sim": d["intra_list_sim"],
            "cold_share": d["cold_share"], "cold_share_delta": cold_delta,
            "valid_impressions": int(r["n_impressions"]), "skipped_impressions": int(r["n_skipped"]),
            "evaluated_lists": int(d["n_lists"]), "excluded_short_lists": int(d["n_short_lists_excluded"]),
        }
        for segment in ["cold", "warm", "mixed"]:
            segment_row = relevance.loc[(weight, rel_scope, segment)]
            baseline = float(relevance.loc[(1.0, rel_scope, segment), metric])
            segment_value = float(segment_row[metric])
            row[f"{segment}_relevance"] = segment_value
            row[f"{segment}_reference"] = baseline
            row[f"{segment}_relative_loss"] = _selection_relative_loss(segment_value, baseline)
            row[f"{segment}_valid_impressions"] = int(segment_row["n_impressions"])
            if segment in segment_limits:
                if not np.isfinite(baseline) or baseline <= 0:
                    failures.append(f"{segment.upper()}_BASELINE_MISSING_OR_NONPOSITIVE")
                elif not _selection_at_least(segment_value, baseline * (1 - segment_limits[segment]), tolerance):
                    failures.append(f"{segment.upper()}_RELEVANCE_GUARD_FAILED_OR_UNAVAILABLE")
        row["qualifies"] = not failures
        row["reason"] = "QUALIFIES" if not failures else "; ".join(failures)
        rows.append(row)
    table = pd.DataFrame(rows).set_index("lambda")
    table["selection_rank"] = pd.Series(pd.NA, index=table.index, dtype="Int64")
    table["is_proposed_candidate"] = False
    qualifiers = table.loc[table["qualifies"]].reset_index()
    qualifiers = qualifiers.sort_values(["diversity_value", "primary_relevance", "lambda"],
        ascending=[policy["objective_ascending"], False, False], kind="stable")
    candidate = None
    if len(qualifiers):
        for priority, weight in enumerate(qualifiers["lambda"], start=1):
            table.loc[weight, "selection_rank"] = priority
        candidate = validate_mmr_lambda(float(qualifiers.iloc[0]["lambda"]))
        table.loc[candidate, "is_proposed_candidate"] = True
    table.attrs["selection_policy"] = deepcopy(policy)
    return table, candidate

#### 2.4.4 Compare relevance-loss budgets on tuning data

Compare loss budgets of 0%, 0.5%, 1%, 2%, and 5% using the completed 14 November tuning summaries. For each budget, apply the same eligible-impression nDCG@10 policy and minimize intra-list similarity, breaking ties by higher relevance and then larger lambda. Show the selected lambda, qualifying-setting count, relevance loss, semantic similarity, category variety, and cold exposure. These are small policy comparisons over existing summaries; no MMR or model scoring is repeated.

The table is a sensitivity analysis, not a rule that the largest budget is best. Choose `MAX_RELATIVE_RELEVANCE_LOSS` in 2.4.1 after reviewing the relevance-diversity trade-off, record the rationale in `SELECTION_POLICY_NOTE`, and run the complete notebook with that choice fixed. The first tuning exploration can reach this table while the final budget is `None`; 2.4.5 then stops with the existing missing-policy message. Do not use Test-day results to choose the budget. `tuning_budget_comparison` is retained for the final CSV exports.


In [ ]:
# --- 2.4.4: Compare candidate loss budgets using existing tuning summaries only. ---
tuning_budget_rows = []
for budget in RELEVANCE_LOSS_BUDGETS:
    # Override only the budget; metric, scope, objective and optional guards remain fixed.
    budget_policy, _ = read_tuning_selection_policy(
        {**globals(), "MAX_RELATIVE_RELEVANCE_LOSS": budget}
    )
    budget_selection, budget_lambda = propose_tuning_lambda(
        tuning_relevance_comparison, tuning_diversity_comparison, tuning_settings, budget_policy,
    )
    row = {"max_relative_relevance_loss": float(budget), "selected_lambda": budget_lambda,
           "qualifying_settings": int(budget_selection["qualifies"].sum())}
    # A budget with no qualifying setting has unavailable selected metrics, rather than zeros.
    for column in ["primary_relevance", "relative_relevance_loss", "intra_list_sim",
                   "unique_categories", "cold_share"]:
        row[column] = budget_selection.loc[budget_lambda, column] if budget_lambda is not None else np.nan
    tuning_budget_rows.append(row)
tuning_budget_comparison = pd.DataFrame(tuning_budget_rows).set_index("max_relative_relevance_loss")
del tuning_budget_rows, budget_selection
print("Tuning-only budget comparison (budget/loss/cold-share values are fractions):")
print(tuning_budget_comparison.to_string(float_format=lambda value: f"{value:.4f}"))


#### 2.4.5 Apply the policy and inspect the proposal

Among qualifying settings, optimize diversity, then prefer higher relevance and larger lambda on exact ties. Print the eligibility/reason table first. If no setting qualifies, stop for a policy decision rather than select a fallback from held-out results.

In [ ]:
# --- 2.4.5: Propose a lambda from tuning results only. ---
tuning_selection_policy, missing_policy_fields = read_tuning_selection_policy(globals())
if missing_policy_fields:
    raise ValueError(f"Fill the policy placeholders in 2.4.1: {missing_policy_fields}")
tuning_selection_table, proposed_tuning_lambda = propose_tuning_lambda(
    tuning_relevance_comparison, tuning_diversity_comparison, tuning_settings, tuning_selection_policy,
)
# Show qualification/rejection reasons even when the declared policy has no solution.
print(tuning_selection_table.to_string(float_format=lambda value: f"{value:.4f}"))
if proposed_tuning_lambda is None:
    raise ValueError("No lambda meets the tuning policy; review the eligibility reasons printed above.")
print("Proposed lambda:", proposed_tuning_lambda)
# A boundary choice can justify a wider tuning grid before fixing the final decision.
if proposed_tuning_lambda == min(tuning_settings["lambdas"]) and len(tuning_settings["lambdas"]) > 1:
    print("Selected the lowest grid value; consider whether the tuning grid should be extended.")

### 2.5 Fix the tuning choice for evaluation

Copy the selected lambda, scaling, model/encoder identifiers, seed, top-k, policy, and selected summaries into `selected_tuning_settings` and `selected_tuning_metrics`. This is the handoff to Section 3; the decision remains fixed during evaluation. Release the large tuning reference frames and MMR arrays while retaining the compact sweep/selection tables and shared embedding lookup for Section 4.

#### 2.5.1 Record selected settings and tuning evidence

Keep experiment metadata and selected summaries for later reporting/export. This ordinary settings record is the handoff to Section 3.

In [ ]:
# --- 2.5.1: Copy the tuning decision and its compact evidence. ---
SELECTED_LAMBDA = float(proposed_tuning_lambda)
selected_tuning_settings = {
    **deepcopy(tuning_settings), "lambda": SELECTED_LAMBDA,
    "relevance_loss_budgets_compared": tuple(RELEVANCE_LOSS_BUDGETS),
    "policy": deepcopy(tuning_selection_policy),
}
selected_tuning_metrics = {
    "relevance": tuning_sweep_summaries[SELECTED_LAMBDA]["relevance"].copy(),
    "diversity": tuning_sweep_summaries[SELECTED_LAMBDA]["diversity"].copy(),
    "decision_row": tuning_selection_table.loc[SELECTED_LAMBDA].to_dict(),
}
print("Selected lambda:", SELECTED_LAMBDA)

#### 2.5.2 Release tuning candidate-level memory

Keep the compact sweep/selection tables and shared vector lookup. Large tuning frames/arrays are no longer needed for evaluation.

In [ ]:
# --- 2.5.2: Release tuning-only candidate tables and array cache. ---

# Compact metric tables and emb_lookup remain available for evaluation.
del baseline_ranked, baseline_top, baseline_slates, mmr_prepared
print("Tuning candidate tables/cache released; summaries and article catalog retained.")

## 3. Held-out evaluation: before versus after

Carry the tuning-selected settings into the labelled evaluation split, construct its relevance-only baseline, and apply the selected MMR setting for the main comparison. Compare both orders using the same metric definitions and population rules as tuning. Export complete per-lambda evaluation rankings in 3.4.3 for Step 6, while keeping the main tuning-selected choice fixed. Retain complete rankings for Section 4's examples and compact results/settings for its figures, conclusions, and exports.

### 3.1 Use the fixed tuning decision

Copy the preceding tuning settings for evaluation, keeping the model/version, encoder, lambda, scaling, top-k, seed, and selection policy fixed. Add the evaluation split/date and configured input paths; later evaluation and reporting cells use this record.

#### 3.1.1 Copy settings for held-out evaluation

Add the evaluation split/date and paths from the active setup. Later ranking cells use this settings copy directly.

In [ ]:
# --- 3.1.1: Carry the tuning-only decision straight into evaluation. ---
evaluation_settings = {
    **deepcopy(selected_tuning_settings), "split": "eval",
    "evaluation_date": str((pd.Timestamp(config.TUNE_DATE) + pd.Timedelta(days=1)).date()),
    "candidate_path": str(EVAL_CANDIDATES_PATH), "score_path": str(EVAL_SCORES_PATH),
}
# Show the held-out locations/date; model, lambda, scaling, and seed were already declared.
print(pd.Series({key: evaluation_settings[key] for key in
                 ["split", "evaluation_date", "candidate_path", "score_path"]}).to_string())

### 3.2 Load evaluation inputs

Read complete processed evaluation candidates, join the selected model's scores, and check article-vector coverage once. Use the same candidate schema and input-reader interfaces as tuning. Sections 3.3-3.4 build the baseline and apply the fixed MMR choice.

#### 3.2.1 Load complete processed evaluation candidates

Read the required columns and inspect counts/a preview. Preserve labels, categories, cold flags, and full slates without resampling.

In [ ]:
# --- 3.2.1: Load and inspect the complete processed evaluation split. ---
eval_candidates = load_candidate_split("eval")
# These counts describe the population; upstream preprocessing owns data cleaning.
print("Evaluation rows:", len(eval_candidates), "| Impressions:", eval_candidates["impression_id"].nunique())
print(eval_candidates.head(5).to_string(index=False))

#### 3.2.2 Attach the selected model's evaluation scores

Use the same strict join as tuning: exactly one score per candidate without silently adding/dropping rows. The upstream producer supplies scores from the recorded model/version. Preserve original scores for AUC.

In [ ]:
# --- 3.2.2: Attach original evaluation scores without retuning/rescaling. ---
eval_scored = load_scored_split("eval", eval_candidates)
del eval_candidates  # The joined frame now contains the complete candidate metadata.
print(eval_scored[KEY_COLUMNS + ["score"]].head(5).to_string(index=False))
print("Score range:", eval_scored["score"].min(), "to", eval_scored["score"].max())

#### 3.2.3 Confirm evaluation article-vector coverage

Check coverage once at the input boundary. The same upstream encoder/catalog must cover both splits. Use `eval_scored`, `emb_lookup`, and `evaluation_settings` directly in the next section, where the slate-size breakdown is printed.

In [ ]:
# --- 3.2.3: Check article-vector coverage before evaluation metrics/MMR. ---
# The full slate-size breakdown will be reported when the baseline ranks are built in 3.3.
require_embedding_coverage(eval_scored, emb_lookup, "eval")

### 3.3 Measure the evaluation relevance-only baseline

Build the held-out reference from the original model scores using the same canonical ordering, seeded score ties, and slate eligibility rule as tuning. Measure relevance, diversity, and cold exposure **before MMR**. The selected lambda/scaling remain fixed in `evaluation_settings`; neither is applied or adjusted in this subsection.

Reuse the existing Section 1 helpers directly. Keep unrounded summaries for the before/after comparison and round only the printed tables.

#### 3.3.1 Build the fixed original-score ranking

Create complete per-impression baseline ranks and top-k lists using the copied evaluation seed/top-k. Only slates with more than k candidates are eligible for later MMR; exact-k and shorter slates retain their reference order. Print the slate-size breakdown and a short ranking preview.

Retain `eval_baseline_ranked`, `eval_baseline_slates`, and `eval_baseline_top` for the next subsection. Release `eval_scored` after building the reference, since the baseline already retains the fields needed downstream.

In [ ]:
# --- 3.3.1: Build the held-out relevance-only reference using the fixed settings. ---
eval_baseline_ranked, eval_baseline_slates, eval_baseline_top = build_relevance_reference(
    eval_scored, k=evaluation_settings["top_k"], seed=evaluation_settings["seed"],
)
# The baseline contains the required scores/labels/metadata; avoid retaining a duplicate.
del eval_scored

print("Eligible (> k):", int(eval_baseline_slates["rerank_eligible"].sum()),
      "| Exact k:", int(eval_baseline_slates["n_candidates"].eq(evaluation_settings["top_k"]).sum()),
      "| Shorter than k:", int(eval_baseline_slates["n_candidates"].lt(evaluation_settings["top_k"]).sum()))
print("Top-list preview before MMR:")
print(eval_baseline_top.head(10).to_string(index=False))

#### 3.3.2 Measure relevance overall and on eligible slates

Calculate original-model-score AUC and fixed-order MRR, nDCG@5, and nDCG@10. Report both all impressions and the eligible subset, each with all/cold/warm/mixed clicked-article segments. An eligible impression keeps its complete slate for relevance metrics, rather than only its displayed top-k.

Follow the shared helper's valid-impression convention: no-click/all-click impressions are skipped and counted. Absent segments remain unavailable rather than becoming zero performance. AUC describes original model scores; later MMR order changes will be assessed with MRR/nDCG.

In [ ]:
# --- 3.3.2: Measure full-slate relevance using the fixed original-score order. ---
eval_baseline_relevance = summarize_reference_relevance(
    eval_baseline_ranked, rank_col="baseline_rank", seed=evaluation_settings["seed"],
)
# Formatting affects only this display; preserve raw values for before/after deltas.
print("Evaluation baseline relevance by scope and clicked-article segment:")
print(eval_baseline_relevance.to_string(float_format=lambda value: f"{value:.4f}"))

#### 3.3.3 Measure top-k diversity and cold exposure

Calculate mean unique categories, mean pairwise signed cosine similarity, and cold share for full top-k lists. Report all full lists separately from the eligible reranking subset. Shorter lists are excluded from these full-list metrics and counted explicitly; exact-k lists contribute only to the all-full-lists scope.

Lower intra-list similarity means greater semantic diversity. Cold share describes exposure, not a bonus used by MMR. Retain the raw table alongside the relevance summary for the fixed-choice comparison in 3.4.

In [ ]:
# --- 3.3.3: Measure baseline full-list diversity and cold-article exposure. ---
eval_baseline_diversity = summarize_reference_diversity(
    eval_baseline_top, lookup=emb_lookup, k=evaluation_settings["top_k"],
)
# Reuse the existing catalog and metric definitions; no files or tuning work are repeated.
print("Evaluation baseline diversity, exposure, and list populations:")
print(eval_baseline_diversity.to_string(float_format=lambda value: f"{value:.4f}"))

### 3.4 Apply the fixed MMR choice and compare evaluation results

Apply the tuning-selected lambda to the held-out baseline for the main evaluation, using the relevance scaling and top-k copied into `evaluation_settings`. Also export the predefined lambda grid for Step 6 comparisons. The additional exports do not select a new lambda or revise the tuning policy using evaluation outcomes. Reuse the Section 1 ranking implementation and existing metric helpers, then calculate before/after changes from the unrounded summaries.

Only eligible slates change. Original model scores, labels, and article metadata remain available in the complete ranking; `rerank_rank` is the final order and `rerank_score` is its negative. Keep the before/after rankings and compact tables for Section 4's examples, plots, and exports.

#### 3.4.1 Prepare evaluation arrays

Prepare the evaluation baseline's grouping, rank, score, and article-vector arrays once for the selected ranking and Step 6 exports. Use the copied top-k value. This stage performs the transformations needed for MMR without reopening input files or repeating preprocessing validation.

In [ ]:
# --- 3.4.1: Prepare the held-out baseline for the selected ranking and Step 6 exports. ---
# Use the top-k copied from the tuning decision; input files are not reopened here.
eval_mmr_prepared = prepare_mmr_inputs(
    eval_baseline_ranked, emb_lookup, k=evaluation_settings["top_k"],
)

#### 3.4.2 Rerank once and inspect the resulting top lists

Apply the selected setting to full evaluation slates. The baseline-best article remains first; subsequent numerical MMR ties use baseline order with the fixed cosine tolerance from 1.4.1. Append unselected articles in baseline order, leaving short/exact-k slates unchanged. A selected lambda of 1 reproduces the baseline exactly.

Extract displayed top-k lists using final ranks, never by sorting intermediate MMR objectives. Retain the prepared array cache for the per-lambda exports in 3.4.3, then release it. Keep the baseline and selected reranked tables for metrics, examples, and comparisons.

In [ ]:
# --- 3.4.2: Apply the tuning-selected lambda exactly once on evaluation data. ---
eval_reranked = apply_mmr(
    eval_mmr_prepared,
    lambda_value=evaluation_settings["lambda"],
    scaling=evaluation_settings["relevance_scaling"],
    progress_every=MMR_PROGRESS_EVERY,
)
# Retain the prepared arrays until the per-lambda Step 6 exports below are complete.

eval_reranked_top = topk(
    eval_reranked.loc[eval_reranked["rerank_rank"].le(evaluation_settings["top_k"])],
    "rerank_rank", k=evaluation_settings["top_k"],
)
print("Reranked top-list preview (original score retained):")
print(eval_reranked_top.head(10).to_string(index=False))

#### 3.4.3 Export full evaluation rankings for Step 6

Export every lambda in the recorded grid (the shared `config.MMR_LAMBDAS` plus the lambda = 1 reference) to `config.SCORES_DIR/nrms_mmr_<lambda>.parquet`. Each file contains exactly `impression_id`, `news_id`, and `score`, with one row per complete evaluation candidate. With the current files, this means 14,085,557 rows per export; the code derives the count from the loaded baseline rather than hard-coding it.

`score = -rerank_rank`: higher means ranked earlier within the impression. The first article receives -1. After greedy top-k selection, remaining articles retain baseline order. Lambda = 1 exports negative baseline ranks. These rank proxies differ from original NRMS logits: Step 6 may calculate final-order AUC from them, while this notebook's `model_score_auc` remains the original-score diagnostic. Even at lambda = 1, resolving original-score ties may change rank-based AUC.

Reuse the selected-lambda result and the prepared evaluation arrays for the other settings. Write one file at a time, preserving the baseline candidate keys without filtering or joining. Check score finiteness at this output boundary, then release the temporary export frame and array cache. The grid is fixed before evaluation; these files support Step 6 comparisons and do not revise `SELECTED_LAMBDA` or the main findings.

The MMR similarity representation is P5's `nrms_main_v2_best.pt:news_encoder`. Step 6's independent diversity representation remains a group/P2 decision and should be used consistently across models. The current notebook's NRMS-space similarity is an optimization diagnostic; it does not replace that independent assessment.


In [ ]:
# --- 3.4.3: Export complete per-lambda evaluation scores for Step 6 without retuning. ---
evaluation_score_dir = Path(config.SCORES_DIR)
evaluation_score_dir.mkdir(parents=True, exist_ok=True)
evaluation_score_exports = []
# Both rank sources align to this complete baseline; no candidate rows are dropped or merged.
evaluation_export_scores = eval_baseline_ranked[KEY_COLUMNS].copy()
for weight in evaluation_settings["lambdas"]:
    if weight == evaluation_settings["lambda"]:
        evaluation_export_scores["score"] = eval_reranked["rerank_score"].to_numpy()
    else:
        evaluation_export_scores["score"] = -run_mmr_ranks(
            eval_mmr_prepared, weight, evaluation_settings["relevance_scaling"],
            progress_every=MMR_PROGRESS_EVERY,
        )
    # Assignment enforces row alignment/length; finite values are required by the handoff.
    if not np.isfinite(evaluation_export_scores["score"].to_numpy()).all():
        raise ValueError(f"Nonfinite exported ranking scores for lambda={weight}.")
    path = evaluation_score_dir / f"nrms_mmr_{weight}.parquet"
    evaluation_export_scores.to_parquet(path, index=False, engine="pyarrow")
    evaluation_score_exports.append({"lambda": float(weight), "path": str(path),
                                     "rows": len(evaluation_export_scores)})
    print(f"Saved lambda={weight:g}: {len(evaluation_export_scores):,} candidate scores to {path}")
# Keep compact file metadata and the selected ranking, rather than all full rankings in memory.
del evaluation_export_scores, eval_mmr_prepared


#### 3.4.4 Measure relevance, diversity, and cold exposure after MMR

Measure full-slate MRR/nDCG from the final order and reuse the baseline's original-model-score AUC, since MMR does not change those scores. Keep the same overall/eligible scopes, clicked-article segments, skipped-impression convention, and full-list exclusions as the baseline.

Compute top-k diversity and cold exposure with the existing helper. The MMR cache was released after the Step 6 exports, so these calculations do not retain a second full article-vector matrix alongside it. Printed rounding does not alter the stored summaries.

In [ ]:
# --- 3.4.4: Measure the final ranking using the same baseline metric definitions. ---
eval_reranked_relevance = summarize_sweep_relevance(
    eval_reranked, "rerank_rank", eval_baseline_relevance,
    {"seed": evaluation_settings["seed"], "cold_column": evaluation_settings["cold_column"]},
)
# Reuse the diversity helper directly; it gathers only vectors needed by the top lists.
eval_reranked_diversity = summarize_reference_diversity(
    eval_reranked_top, lookup=emb_lookup, k=evaluation_settings["top_k"],
)
print("Relevance after MMR (AUC still describes original model scores):")
print(eval_reranked_relevance.to_string(float_format=lambda value: f"{value:.4f}"))
print("Diversity and cold exposure after MMR:")
print(eval_reranked_diversity.to_string(float_format=lambda value: f"{value:.4f}"))

#### 3.4.5 Assemble before/after comparisons for reporting

Keep `evaluation_relevance_comparison` and `evaluation_diversity_comparison` as the complete baseline/MMR summaries, including population counts. Build the long-form `evaluation_results` table with one row per metric/scope/segment and columns for baseline, MMR, signed delta, relative percentage change, and interpretation. Its attributes retain the evaluation settings for Section 4; no files are exported here.

Reuse the tuning comparison's arithmetic: delta = MMR minus baseline, and relative change = `100 * delta / abs(baseline)`. Missing metrics and zero baselines retain unavailable percentages with explanatory metric-availability labels. These labels identify undefined arithmetic. Section 3.5 prints the overview and segment comparisons, so the complete table is retained here without repeating those rows.

Higher MRR/nDCG and category variety are favorable; lower intra-list similarity means greater semantic diversity. Cold share is an exposure diagnostic. AUC is unchanged by construction. The held-out results describe the fixed choice and must not be used to select another lambda.

In [ ]:
# --- 3.4.5: Assemble full-precision held-out before/after reporting tables. ---
evaluation_relevance_comparison = pd.concat(
    {"baseline": eval_baseline_relevance, "mmr": eval_reranked_relevance}, names=["ranking"],
)
evaluation_diversity_comparison = pd.concat(
    {"baseline": eval_baseline_diversity, "mmr": eval_reranked_diversity}, names=["ranking"],
)

# Build compact deltas without repeating any ranking, file loading, or metric work.
evaluation_rows = []
for family, before, after, metrics in [
    ("relevance", eval_baseline_relevance, eval_reranked_relevance,
     ["model_score_auc", "mrr", "ndcg@5", "ndcg@10"]),
    ("diversity", eval_baseline_diversity, eval_reranked_diversity,
     ["unique_categories", "intra_list_sim", "cold_share"]),
]:
    for key in before.index:
        scope, segment = key if family == "relevance" else (key, "list")
        for metric in metrics:
            reference, value = before.loc[key, metric], after.loc[key, metric]
            delta, relative, availability = _tuning_metric_change(value, reference)
            interpretation = {
                "model_score_auc": "fixed_model_score_metric",
                "intra_list_sim": "lower_is_more_diverse",
                "cold_share": "exposure_diagnostic",
                "unique_categories": "higher_is_more_varied",
            }.get(metric, "higher_is_better")
            evaluation_rows.append({
                "family": family, "scope": scope, "segment": segment, "metric": metric,
                "baseline": reference, "mmr": value, "delta": delta,
                "relative_change_pct": relative, "change_status": availability,
                "interpretation": interpretation,
            })
evaluation_results = pd.DataFrame(evaluation_rows).set_index(["family", "scope", "segment", "metric"])
evaluation_results.attrs["experiment_settings"] = deepcopy(evaluation_settings)
evaluation_results.attrs["relative_change_definition"] = "100 * (mmr - baseline) / abs(baseline)"
del evaluation_rows

### 3.5 Summarize held-out findings and prepare reporting inputs

Organize the completed evaluation results into readable reporting views: overall/eligible performance, clicked-article segment diagnostics, population counts, and experiment metadata. Reuse the raw summaries from 3.3-3.4 without rereading files, reranking, recalculating metrics, or changing the tuning-selected decision.

Keep the complete `evaluation_results` table as the detailed source. The smaller views below support Section 4's plots, examples, and exports; they retain the original metric values and populations. No files are exported in this subsection.

#### 3.5.1 Build a concise overall and eligible-slate overview

Show baseline, MMR, signed delta, and relative percentage change for original-score AUC, MRR, nDCG@5/10, unique categories, intra-list similarity, and cold share. Keep the original source scope in each row: overall relevance includes all slates, whereas overall diversity uses only full top-k lists. Eligible-slate results use that same eligible subset for both metric families.

AUC remains a fixed model-score diagnostic; higher cold share is an exposure observation. Preserve metric-availability labels for undefined changes and keep numeric values unrounded until printing.

In [ ]:
# --- 3.5.1: Create a compact overview from the already calculated metric changes. ---
evaluation_overview_rows = []
for report_scope, relevance_scope, diversity_scope in [
    ("overall", "all_impressions", "all_full_topk_lists"),
    ("eligible_impressions", "eligible_impressions", "eligible_impressions"),
]:
    # Keep the original population scope visible instead of implying identical denominators.
    for family, source_scope, segment, metrics in [
        ("relevance", relevance_scope, "all", ["model_score_auc", "mrr", "ndcg@5", "ndcg@10"]),
        ("diversity", diversity_scope, "list", ["unique_categories", "intra_list_sim", "cold_share"]),
    ]:
        for metric in metrics:
            values = evaluation_results.loc[(family, source_scope, segment, metric)].to_dict()
            evaluation_overview_rows.append({
                "report_scope": report_scope, "family": family, "metric": metric,
                "source_scope": source_scope, **values,
            })
evaluation_overview = pd.DataFrame(evaluation_overview_rows).set_index(["report_scope", "family", "metric"])
del evaluation_overview_rows

print("Held-out overview: overall and eligible-impression results")
print(evaluation_overview.to_string(float_format=lambda value: f"{value:.4f}"))

#### 3.5.2 Inspect clicked-article segments and metric populations

Show before/after MRR and nDCG for cold, warm, and mixed clicked-article segments, with valid/skipped impression counts alongside each result. Segments describe clicked articles within complete impressions; they do not filter the candidate slate by cold status. Missing segments remain unavailable.

Keep the complete baseline/MMR population counts in the comparison tables; they were already printed with the baseline and post-MMR metrics, so do not print them again here. Short slates contribute to overall relevance but are excluded from full-list diversity; exact-k lists contribute to overall diversity but are ineligible for reranking. Counts explain the denominators rather than trigger another validation pass.

In [ ]:
# --- 3.5.2: Attach existing population counts to the cold/warm/mixed relevance views. ---
evaluation_segment_summary = evaluation_results.xs("relevance", level="family")
evaluation_segment_summary = evaluation_segment_summary.loc[
    evaluation_segment_summary.index.get_level_values("segment").isin(["cold", "warm", "mixed"])
    & evaluation_segment_summary.index.get_level_values("metric").isin(["mrr", "ndcg@5", "ndcg@10"])
].copy()
# Both rankings use the same impressions; these baseline counts describe each metric row.
evaluation_segment_summary = evaluation_segment_summary.join(
    eval_baseline_relevance[["n_impressions", "n_skipped"]], on=["scope", "segment"],
)
print("Clicked-article segment relevance and valid/skipped populations:")
print(evaluation_segment_summary.to_string(float_format=lambda value: f"{value:.4f}"))

#### 3.5.3 Collect reporting metadata for Section 4

Record the fixed evaluation settings and simple population totals beside the tables. Preserve the upstream model/encoder identifiers, input paths, seed, lambda, scaling, and tuning-only selection policy already recorded in `evaluation_settings`. These values accompany the exported tables and report.

Section 4 uses `evaluation_overview`, `evaluation_segment_summary`, `evaluation_results`, the full comparison tables, and `evaluation_report_metadata` for presentation/export. The retained baseline/reranked top lists support examples. Interpret held-out changes as observations from one fixed choice; do not revise lambda using this table or claim statistical significance from aggregate changes alone.

In [ ]:
# --- 3.5.3: Package the fixed run's settings, population totals, and metric definitions. ---
evaluation_report_metadata = {
    "experiment_settings": deepcopy(evaluation_settings),
    # Step 6 reads the full candidate scores; its AUC uses final order rather than NRMS logits.
    "step6_handoff": {
        "files": deepcopy(evaluation_score_exports),
        "score_definition": "negative complete final rank within each impression; higher score means ranked earlier",
        "auc_definition": "Step 6 rank-score AUC is distinct from this notebook's original model_score_auc; lambda=1 resolves raw-score ties",
        "similarity_encoder": evaluation_settings["embedding_encoder_id"],
        "independent_diversity_embedding": "Group/P2 to select a different representation for consistent Step 6 evaluation",
    },
    "population": {
        "candidate_rows": len(eval_baseline_ranked),
        "impressions": len(eval_baseline_slates),
        "eligible_impressions": int(eval_baseline_slates["rerank_eligible"].sum()),
        "exact_k_impressions": int(eval_baseline_slates["n_candidates"].eq(evaluation_settings["top_k"]).sum()),
        "short_impressions": int(eval_baseline_slates["n_candidates"].lt(evaluation_settings["top_k"]).sum()),
    },
    # Include denominator and unit conventions so later exported tables remain interpretable.
    "metric_definitions": {
        "relevance": "complete slates; shared no-click/all-click skipping convention",
        "segments": "cold/warm/mixed status of clicked articles within each impression",
        "diversity": "full top-k lists only; eligible subset reported separately",
        "model_score_auc": "original model scores; unchanged by MMR",
        "delta": "MMR minus baseline in raw metric units",
        "relative_change_pct": "100 * delta / abs(baseline); unavailable for zero/missing baseline",
        "cold_share": "fraction of displayed articles marked cold; exposure diagnostic",
        "intra_list_sim": "mean pairwise signed cosine in the NRMS space used by MMR; optimization diagnostic, lower is more diverse",
    },
}

### 3.6 Interpret the fixed-choice results and save compact summaries

Explain the observed held-out trade-off using the relevance metric, diversity objective, and reporting scope declared by the tuning policy. Describe cold/warm/mixed results with their population counts and retain limitations beside the interpretation. Reuse existing summaries; do not rerank, recalculate metrics, or change the selected lambda/policy.

Save ordinary compact CSV/JSON outputs under `Path(config.OUTPUT_DIR) / "step5_evaluation_summary"`. These files preserve the completed evaluation for inspection. Section 4 uses the retained in-memory results to produce plots, examples, conclusions, and final presentation exports.

#### 3.6.1 Describe the observed trade-off and segment differences

Read the chosen policy's primary relevance metric and diversity objective from the held-out results. Build factual before/after findings and directions for the primary metrics and cold/warm/mixed clicked-article segments in that policy scope. Retain this narrative for the JSON export rather than reprinting values already shown in 3.5. A missing segment is described as unavailable rather than assigned zero performance.

The direction of a raw change is not always its interpretation: lower intra-list similarity means more semantic diversity; higher category count means more category variety. Cold share is reported as exposure; print its absolute change in percentage points as an additional unit view. The relevance-loss budget remains a tuning criterion, not a held-out acceptance gate or a reason to select another lambda.

In [ ]:
# --- 3.6.1: Describe the already measured fixed-choice trade-off and primary segments. ---
def describe_evaluation_metric(metric, values):
    """Format observed changes without inventing findings for unavailable metrics."""
    # Missing/zero-reference percentages already have their mathematical meaning in the tables.
    if pd.isna(values["baseline"]) or pd.isna(values["mmr"]):
        return f"{metric}: unavailable for this population."
    delta = values["delta"]
    direction = "increased" if delta > 0 else "decreased" if delta < 0 else "unchanged"
    return (f"{metric}: {values['baseline']:.4f} -> {values['mmr']:.4f}; "
            f"{direction} (delta {delta:+.6f}).")


# The tuning policy fixes which held-out scope/metrics we emphasize; it is never reapplied here.
eval_reporting_policy = evaluation_settings["policy"]
eval_primary_metric = eval_reporting_policy["relevance_metric"]
eval_primary_relevance = evaluation_results.loc[
    ("relevance", eval_reporting_policy["relevance_scope"], "all", eval_primary_metric)
]
eval_primary_diversity = evaluation_results.loc[
    ("diversity", eval_reporting_policy["diversity_scope"], "list", eval_reporting_policy["diversity_metric"])
]
eval_primary_segments = evaluation_segment_summary.xs(
    (eval_reporting_policy["relevance_scope"], eval_primary_metric), level=["scope", "metric"],
)
eval_cold_exposure = evaluation_results.loc[
    ("diversity", eval_reporting_policy["diversity_scope"], "list", "cold_share")
]
evaluation_findings = {
    "scope": eval_reporting_policy["scope"],
    "relevance": describe_evaluation_metric(eval_primary_metric, eval_primary_relevance),
    "diversity": describe_evaluation_metric(eval_reporting_policy["diversity_metric"], eval_primary_diversity),
    "cold_exposure": describe_evaluation_metric("cold_share", eval_cold_exposure),
    "segments": {},
}
for segment, values in eval_primary_segments.iterrows():
    # Segment counts explain how much evidence each reported change represents.
    evaluation_findings["segments"][segment] = (
        describe_evaluation_metric(eval_primary_metric, values)
        + f" Valid impressions: {int(values['n_impressions']):,}; skipped: {int(values['n_skipped']):,}."
    )
# The tables above already show these numeric results; retain narrative findings for JSON.
# Percentage points are a new unit view of cold exposure, so show that change once here.
if pd.notna(eval_cold_exposure["delta"]):
    print(f"Cold-share change ({eval_reporting_policy['scope']}): "
          f"{100 * eval_cold_exposure['delta']:+.3f} percentage points.")

#### 3.6.2 State the interpretation limits

- These are offline, click-based observations for one held-out split and one tuning-selected setting. Aggregate differences alone do not establish statistical significance or online user satisfaction; paired uncertainty estimates would be needed to assess statistical uncertainty.
- Cold/warm/mixed labels describe the **clicked articles within an impression**. They are not separate candidate pools. Interpret segment differences alongside valid/skipped counts, especially for small segments.
- Overall relevance includes short and exact-k slates whose ranks MMR leaves unchanged. Overall diversity includes full top-k lists; eligible-only results show the population MMR actually changes. These denominators are intentionally different.
- Lower semantic similarity does not guarantee greater relevance. Category variety and embedding-based semantic diversity describe different aspects of a list. More cold exposure does not by itself demonstrate fairness or improved cold-article relevance.
- Original-score AUC is unchanged by construction. MRR/nDCG assess the new order, while diversity/cold share describe the displayed lists. No claim about calibration or original-score AUC improvement follows from MMR.
- Interpretation depends on the upstream split, score model/version, encoder, and cold-flag definitions. Record their identifiers and use the same artifacts/settings for reproducible reruns. Report held-out outcomes without retuning lambda or revising the policy on this split.

These limits accompany the saved findings. Section 4 uses examples and plots to explain the observations without overstating them.

Semantic similarity is measured in the same NRMS space MMR optimizes. Step 6 should use a group-agreed different embedding for an independent diversity assessment, while retaining consistent evaluation settings across all models.

In [ ]:
# --- 3.6.2: Retain the main interpretation limits alongside the numeric results. ---
evaluation_limitations = [
    "Semantic diversity here uses the NRMS representation optimized by MMR; Step 6 needs a group-agreed different embedding for independent evaluation.",
    "Offline results on one held-out split; aggregate changes do not establish statistical significance or online benefit.",
    "Segments describe clicked-article cold status; compare them with valid/skipped impression counts.",
    "Overall relevance and full-list diversity have different denominators; eligible-only results are reported separately.",
    "Semantic/category diversity and cold exposure do not guarantee relevance gains or fairness.",
    "Original-score AUC is fixed; ranking changes are assessed with MRR/nDCG.",
    "Results depend on upstream model/encoder/split/cold definitions; evaluation is not used for retuning.",
]
# The limitations are explained in the Markdown above and saved in run.json below.

#### 3.6.3 Save compact evaluation tables and the run description

Write five CSVs (`metrics.csv`, `relevance.csv`, `diversity.csv`, `overview.csv`, `segments.csv`) and `run.json` to `step5_evaluation_summary` under the active setup's output folder. The CSV index levels retain scope/segment/ranking identifiers; numeric summaries are saved without display rounding. Relevance/diversity tables also preserve the population counts used by the metrics.

`run.json` contains the recorded experiment settings, population totals, metric/unit definitions, findings, limitations, and Python/NumPy/pandas versions. Save metadata explicitly because CSV does not retain DataFrame attributes. These compact summary files omit candidate-level rankings and embeddings. Full candidate rankings have already been exported in 3.4.3, and selected before/after rankings remain in memory for Section 4's examples. `run.json` also records the Step 6 file paths, row counts, score meaning, similarity encoder, and independent-evaluation handoff. Running the notebook again replaces these fixed-name summaries for that run.

In [ ]:
# --- 3.6.3: Save ordinary compact outputs using the active shared configuration. ---
import json
from platform import python_version

evaluation_summary_dir = Path(config.OUTPUT_DIR) / "step5_evaluation_summary"
evaluation_summary_dir.mkdir(parents=True, exist_ok=True)
# These are small completed summaries; no candidate rows or embedding matrices are exported.
for filename, table in {
    "metrics.csv": evaluation_results,
    "relevance.csv": evaluation_relevance_comparison,
    "diversity.csv": evaluation_diversity_comparison,
    "overview.csv": evaluation_overview,
    "segments.csv": evaluation_segment_summary,
}.items():
    table.to_csv(evaluation_summary_dir / filename)

evaluation_summary_record = {
    **deepcopy(evaluation_report_metadata),
    "findings": deepcopy(evaluation_findings),
    "limitations": list(evaluation_limitations),
    "software_versions": {"python": python_version(), "numpy": np.__version__, "pandas": pd.__version__},
}
# Serialize settings explicitly; CSV drops attrs, and this is an ordinary output, not a recovery file.
with (evaluation_summary_dir / "run.json").open("w", encoding="utf-8") as summary_file:
    json.dump(evaluation_summary_record, summary_file, indent=2, ensure_ascii=False, allow_nan=False)
print("Saved five compact CSV tables and run.json to:", evaluation_summary_dir)

## 4. Trade-off plots, example slates, and conclusions

Plot the tuning relevance-diversity trade-off and identify the recorded operating point. Present held-out before/after results, then inspect seeded illustrative slates to explain article movements and local diversity/exposure changes.

Use the existing full-precision summaries and fixed settings from Sections 2-3. Summarize observations and limitations, distinguishing semantic/category diversity and cold exposure from relevance or fairness. Export tables, figures, examples, conclusions, and provenance through the shared configuration paths.

### 4.1 Prepare presentation settings and plotting views

Define consistent figure styling, labels, ordering, and output paths, then create small views of the results already calculated in Sections 2-3. Read the primary metrics and scopes from the recorded tuning policy; the selected lambda remains `evaluation_settings["lambda"]`.

These cells do not reopen datasets, rerank candidates, recalculate metrics, or change the selected settings. Values remain unrounded, cold share remains a fraction, and unavailable values remain missing. Later plot cells will format labels/percentage axes without changing the source data. This stage configures destinations but creates no folders, figures, or output files.

#### 4.1.1 Define figure styling, labels, and output destinations

Use Matplotlib with a consistent blue baseline/orange MMR palette, readable font sizes, and PNG/PDF export settings. Keep figure dimensions, scope/segment order, and display precision together so later plots share the same conventions. The style update affects plotting only.

Construct all paths from the active setup's `config.OUTPUT_DIR`. Final tuning/evaluation CSVs use `results_step5_tuning.csv` and `results_step5.csv` under `config.OUTPUT_DIR`; figures, examples, conclusions, and reporting metadata use a separate `step5_report` folder. The compact exports already written in 3.6 remain in their own folder.

In [ ]:
# --- 4.1.1: Configure presentation styling and paths without touching experiment results. ---
REPORT_DIR = Path(config.OUTPUT_DIR) / "step5_report"
FIGURE_DIR = REPORT_DIR / "figures"
TUNING_RESULTS_PATH = Path(config.OUTPUT_DIR) / "results_step5_tuning.csv"
EVAL_RESULTS_PATH = Path(config.OUTPUT_DIR) / "results_step5.csv"
EXAMPLE_SLATES_PATH = REPORT_DIR / "example_slates.csv"
REPORT_METADATA_PATH = REPORT_DIR / "report_metadata.json"
REPORT_CONCLUSIONS_PATH = REPORT_DIR / "conclusions.md"

# These settings affect presentation/export precision, never the stored metric values.
FIGURE_FORMATS = ("png", "pdf")
FIGURE_EXPORT_DPI = 300
REPORT_DISPLAY_DECIMALS = 4
FIGURE_SIZES = {
    "tradeoff": (7, 5), "lambda_diagnostics": (12, 4.5),
    "evaluation": (12, 7), "segments": (11, 5.5),
}
REPORT_COLORS = {
    "baseline": "#0072B2", "mmr": "#D55E00",
    "qualifying": "#009E73", "rejected": "#808080",
}
REPORT_STYLE = {
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.titlesize": 11, "axes.labelsize": 10, "legend.fontsize": 9,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.alpha": 0.2,
    "figure.dpi": 110, "savefig.dpi": FIGURE_EXPORT_DPI, "pdf.fonttype": 42,
}
plt.rcParams.update(REPORT_STYLE)

# Labels retain the metric definitions; percentage formatting for cold share comes at plotting time.
METRIC_LABELS = {
    "model_score_auc": "Original-score AUC", "mrr": "MRR",
    "ndcg@5": "nDCG@5", "ndcg@10": "nDCG@10",
    "unique_categories": "Unique categories",
    "intra_list_sim": "Mean pairwise cosine similarity", "cold_share": "Cold-article share",
}
REPORT_SCOPE_ORDER = ("overall", "eligible_impressions")
SEGMENT_ORDER = ("cold", "warm", "mixed")
REPORT_SCOPE_LABELS = {
    "overall": "Overall", "all_impressions": "All impressions",
    "all_full_topk_lists": f"All full top-{evaluation_settings['top_k']} lists",
    "eligible_impressions": f"Eligible impressions (> {evaluation_settings['top_k']} candidates)",
}

#### 4.1.2 Prepare the tuning trade-off and lambda-diagnostic view

Take the primary relevance metric from the policy's relevance scope/all-segment summary, and join the diversity metrics from the policy's diversity scope by lambda. Retain valid/skipped impression counts, full-list counts, and existing policy qualification outcomes for figure captions/markers. Detailed qualification reasons remain in `tuning_selection_table`. Do not apply the policy again.

`tuning_plot_data` has one row per tested lambda. Its generic `relevance` and `diversity` columns correspond to the policy-selected metrics; the individual category/similarity/cold-share columns remain available for diagnostics. Baseline and selected markers use the corresponding lambda rows directly; both identify the same row when lambda = 1 is selected.

`tuning_relevance_floor` is the tuning budget's horizontal reference line. It comes from the tuning baseline and recorded loss fraction, never from held-out performance. The scope note keeps overall relevance versus full-list diversity denominators explicit.

In [ ]:
# --- 4.1.2: Assemble small tuning plot inputs by lambda from the existing summaries. ---
report_policy = evaluation_settings["policy"]
report_relevance_metric = report_policy["relevance_metric"]
report_diversity_metric = report_policy["diversity_metric"]

# Select the exact scopes used for the tuning decision; retain raw values and population counts.
tuning_plot_data = tuning_relevance_comparison.xs(
    (report_policy["relevance_scope"], "all"), level=["scope", "segment"],
)[[report_relevance_metric, "n_impressions", "n_skipped"]].rename(
    columns={report_relevance_metric: "relevance"},
).join(
    tuning_diversity_comparison.xs(report_policy["diversity_scope"], level="scope")[[
        "unique_categories", "intra_list_sim", "cold_share",
        "n_lists", "n_short_lists_excluded",
    ]],
).join(
    tuning_selection_table[["qualifies"]],
).reindex(selected_tuning_settings["lambdas"])
tuning_plot_data["diversity"] = tuning_plot_data[report_diversity_metric]

# Store the plot reference line and denominator wording, without selecting another operating point.
tuning_relevance_floor = (
    tuning_plot_data.loc[1.0, "relevance"] * (1 - report_policy["max_relative_relevance_loss"])
)
report_scope_note = (
    f"Relevance: {REPORT_SCOPE_LABELS[report_policy['relevance_scope']]}; "
    f"diversity: {REPORT_SCOPE_LABELS[report_policy['diversity_scope']]}"
)

#### 4.1.3 Prepare held-out metric and segment views

Reuse the existing overview for the six before/after plotted metrics: MRR, nDCG@5/10, category count, semantic similarity, and cold share. Keep original-score AUC in the analytical results/exports, but omit it from the improvement plots because reranking leaves it unchanged. Preserve the original `source_scope` so overall relevance is not confused with full-list diversity.

`evaluation_plot_data` contains the overall/eligible metric rows. `segment_plot_data` contains the policy's primary relevance metric for cold/warm/mixed segments in both scopes, including valid/skipped counts. Do not replace missing segments with zeros. Overall population details also remain available in `eval_baseline_relevance` and `eval_baseline_diversity` for later captions.

In [ ]:
# --- 4.1.3: Reuse compact held-out comparisons rather than recomputing any metrics. ---
REPORT_METRIC_ORDER = (
    "mrr", "ndcg@5", "ndcg@10", "unique_categories", "intra_list_sim", "cold_share",
)
# Retain population scopes and raw units for the axis labels and captions.
evaluation_plot_data = evaluation_overview.loc[
    evaluation_overview.index.get_level_values("metric").isin(REPORT_METRIC_ORDER)
].reset_index()

segment_plot_data = evaluation_segment_summary.xs(
    report_relevance_metric, level="metric",
).reset_index()

#### 4.1.4 Inspect the new presentation inputs

Print only the new report destinations and plotting-view sizes. The metric values, policy decision, and population tables were already shown in earlier sections, so they are not printed again. Section 4.2 will use these inputs for the tuning figures; 4.3 will use them for held-out comparisons.

In [ ]:
# --- 4.1.4: Report only newly configured destinations and small plotting-view sizes. ---
print("Report folder:", REPORT_DIR)
print("Final CSV destinations:", TUNING_RESULTS_PATH, "and", EVAL_RESULTS_PATH)
# These counts describe newly prepared plot inputs, not another data-validation pass.
print("Plot inputs:", len(tuning_plot_data), "tuning points;",
      len(evaluation_plot_data), "held-out metric rows;", len(segment_plot_data), "segment rows.")

### 4.2 Visualize the tuning relevance-diversity trade-off

Use the tuning-only plotting view from 4.1 to explain the operating point selected in Section 2. The figures do not select another lambda or use held-out results to change the policy. Reuse raw metric values, the recorded qualification results, and the tuning relevance floor; no ranking or metric calculations are repeated.

The first figure relates the policy's relevance metric to its diversity objective. The second shows category variety, semantic similarity, and cold exposure across the tested lambda grid. Retain the figure objects and captions for Section 4.6's PNG/PDF exports; display them here without saving files or reprinting numeric tables.

#### 4.2.1 Plot the primary relevance-diversity trade-off

Plot one point per tested lambda using the exact scopes of the declared policy. Green circles qualify under all policy constraints; grey crosses do not. A blue outlined square marks the relevance-only baseline, and an orange star marks the already-selected operating point. If lambda = 1 was selected, both markers identify the same point. Coincident metric coordinates share a lambda annotation so identical points do not produce stacked labels.

Show the tuning relevance-loss budget as a horizontal line. This line displays only the primary relevance constraint; point qualification also reflects any declared cold-share or segment constraints. Label whether lower similarity or higher category variety is desirable, and include valid/skipped impression and full-list counts in the caption. Undefined metric points are omitted from the scatter, never replaced with zeros.

In [ ]:
# --- 4.2.1: Display the tuning trade-off without changing the selected operating point. ---
tuning_tradeoff_fig, tuning_tradeoff_ax = plt.subplots(figsize=FIGURE_SIZES["tradeoff"])
# Drop missing coordinates only from this small drawing view; analytical results stay unchanged.
tuning_tradeoff_points = tuning_plot_data.dropna(subset=["relevance", "diversity"])
for qualifies, color, marker, label in [
    (True, REPORT_COLORS["qualifying"], "o", "Meets tuning policy"),
    (False, REPORT_COLORS["rejected"], "x", "Outside tuning policy"),
]:
    points = tuning_tradeoff_points.loc[tuning_tradeoff_points["qualifies"].eq(qualifies)]
    if not points.empty:
        tuning_tradeoff_ax.scatter(points["diversity"], points["relevance"],
                                   color=color, marker=marker, s=55, label=label, zorder=3)

# An outline and a star remain visible together if the selected lambda is the baseline.
tuning_baseline_point = tuning_plot_data.loc[1.0]
tuning_selected_point = tuning_plot_data.loc[evaluation_settings["lambda"]]
tuning_tradeoff_ax.scatter(
    [tuning_baseline_point["diversity"]], [tuning_baseline_point["relevance"]],
    marker="s", s=240, facecolors="none", edgecolors=REPORT_COLORS["baseline"],
    linewidths=1.8, label="Baseline (lambda = 1)", zorder=4,
)
tuning_tradeoff_ax.scatter(
    [tuning_selected_point["diversity"]], [tuning_selected_point["relevance"]],
    marker="*", s=160, color=REPORT_COLORS["mmr"], edgecolors="black", linewidths=0.5,
    label=f"Selected (lambda = {evaluation_settings['lambda']:g})", zorder=5,
)
tuning_tradeoff_ax.axhline(
    tuning_relevance_floor, color="0.35", linestyle="--", linewidth=1,
    label=f"Relevance floor ({100 * report_policy['max_relative_relevance_loss']:g}% loss budget)",
)

# Group identical coordinates and direct labels inward; do not jitter measured metric values.
label_x_middle = (tuning_tradeoff_points["diversity"].min() + tuning_tradeoff_points["diversity"].max()) / 2
label_y_middle = (tuning_tradeoff_points["relevance"].min() + tuning_tradeoff_points["relevance"].max()) / 2
for (diversity, relevance), coincident in tuning_tradeoff_points.groupby(
    ["diversity", "relevance"], sort=False,
):
    offset = (-8 if diversity >= label_x_middle else 8, -16 if relevance >= label_y_middle else 10)
    horizontal_alignment = "right" if offset[0] < 0 else "left"
    annotation = "lambda = " + ", ".join(f"{weight:g}" for weight in coincident.index)
    tuning_tradeoff_ax.annotate(annotation, (diversity, relevance), xytext=offset,
                                textcoords="offset points", fontsize=8, ha=horizontal_alignment,
                                bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.8})

diversity_direction = "lower = more diverse" if report_diversity_metric == "intra_list_sim" else "higher = more varied"
tuning_tradeoff_ax.set_xlabel(f"{METRIC_LABELS[report_diversity_metric]} ({diversity_direction})")
tuning_tradeoff_ax.set_ylabel(f"{METRIC_LABELS[report_relevance_metric]} (higher is better)")
tuning_tradeoff_ax.set_title("Tuning relevance-diversity trade-off")
tuning_tradeoff_ax.ticklabel_format(axis="both", style="plain", useOffset=False)
tuning_tradeoff_ax.margins(x=0.12, y=0.12)
tuning_tradeoff_ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2, frameon=False)
tuning_tradeoff_fig.subplots_adjust(left=0.13, right=0.97, top=0.90, bottom=0.36)

# Denominators come from the fixed tuning population, not the held-out evaluation summaries.
tuning_tradeoff_caption = (
    f"Relevance: {REPORT_SCOPE_LABELS[report_policy['relevance_scope']]} | "
    f"valid: {int(tuning_baseline_point['n_impressions']):,}; skipped: {int(tuning_baseline_point['n_skipped']):,}.\n"
    f"Diversity: {REPORT_SCOPE_LABELS[report_policy['diversity_scope']]} | "
    f"full lists: {int(tuning_baseline_point['n_lists']):,}; short lists excluded: {int(tuning_baseline_point['n_short_lists_excluded']):,}.\n"
    "Markers include all policy constraints. Coincident points share lambda labels."
)
tuning_tradeoff_fig.text(0.02, 0.02, tuning_tradeoff_caption, ha="left", va="bottom", fontsize=8)
plt.show()

#### 4.2.2 Plot diversity and cold exposure across lambda

Use the same tuning diversity scope in three panels: mean unique categories, mean pairwise signed cosine similarity, and cold-article share. Order lambda numerically and connect tested points only; the lines do not establish untested performance or a Pareto frontier. Keep the baseline/selected markers and policy qualification styling consistent with the trade-off figure.

Cosine keeps its signed values. Cold share remains a fraction in the source table and is formatted as percentages on its axis. A metric unavailable for the entire population is labelled unavailable rather than drawn as zero. Each panel has its own metric scale, and the caption retains full-list exclusions. These diagnostic views describe the selected policy's outcomes without adding another selection criterion.

In [ ]:
# --- 4.2.2: Display tuning category, semantic-diversity, and exposure diagnostics. ---
tuning_diagnostics_data = tuning_plot_data.sort_index()
tuning_diagnostics_fig, tuning_diagnostics_axes = plt.subplots(
    1, 3, figsize=FIGURE_SIZES["lambda_diagnostics"], sharex=True,
)
for axis, metric, title in zip(
    tuning_diagnostics_axes,
    ["unique_categories", "intra_list_sim", "cold_share"],
    ["Category variety", "Semantic similarity", "Cold-article exposure"],
):
    # Missing metric values remain gaps; an entirely undefined metric gets an explicit label.
    if tuning_diagnostics_data[metric].notna().any():
        axis.plot(tuning_diagnostics_data.index, tuning_diagnostics_data[metric], color="0.70", linewidth=1, zorder=1)
        for qualifies, color, marker, label in [
            (True, REPORT_COLORS["qualifying"], "o", "Meets tuning policy"),
            (False, REPORT_COLORS["rejected"], "x", "Outside tuning policy"),
        ]:
            points = tuning_diagnostics_data.loc[tuning_diagnostics_data["qualifies"].eq(qualifies)]
            if not points.empty:
                axis.scatter(points.index, points[metric], color=color, marker=marker, s=35, label=label, zorder=3)
        axis.scatter([1.0], [tuning_baseline_point[metric]], marker="s", s=180, facecolors="none",
                     edgecolors=REPORT_COLORS["baseline"], linewidths=1.6, label="Baseline (lambda = 1)", zorder=4)
        axis.scatter([evaluation_settings["lambda"]], [tuning_selected_point[metric]], marker="*", s=110,
                     color=REPORT_COLORS["mmr"], edgecolors="black", linewidths=0.5,
                     label=f"Selected (lambda = {evaluation_settings['lambda']:g})", zorder=5)
    else:
        axis.text(0.5, 0.5, "Unavailable for this population", transform=axis.transAxes,
                  ha="center", va="center", fontsize=9, color="0.4")
    axis.set_title(title)
    axis.set_xlabel("Relevance weight (lambda)")
    axis.set_ylabel(METRIC_LABELS[metric])
    axis.set_xticks(tuning_diagnostics_data.index, [f"{weight:g}" for weight in tuning_diagnostics_data.index])
    axis.margins(x=0.12, y=0.12)
    if metric == "cold_share":
        axis.yaxis.set_major_formatter(PercentFormatter(xmax=1))
    else:
        axis.ticklabel_format(axis="y", style="plain", useOffset=False)

# Use one shared legend rather than repeating it inside each panel.
handles, labels = tuning_diagnostics_axes[0].get_legend_handles_labels()
tuning_diagnostics_fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, 0.14),
                              ncol=4, frameon=False)
tuning_diagnostics_fig.suptitle("Tuning diagnostics across the tested lambda grid", y=0.98)
tuning_diagnostics_fig.subplots_adjust(left=0.07, right=0.98, top=0.80, bottom=0.33, wspace=0.32)
tuning_diagnostics_caption = (
    f"Tuning diversity: {REPORT_SCOPE_LABELS[report_policy['diversity_scope']]} | "
    f"full lists: {int(tuning_baseline_point['n_lists']):,}; short lists excluded: {int(tuning_baseline_point['n_short_lists_excluded']):,}.\n"
    "Each panel uses its own metric scale. Lines connect tested points only; cold share is an exposure diagnostic."
)
tuning_diagnostics_fig.text(0.02, 0.025, tuning_diagnostics_caption, ha="left", va="bottom", fontsize=8)
plt.show()

### 4.3 Visualize held-out performance before and after MMR

Use the fixed-choice evaluation views prepared in 4.1. Compare baseline and MMR relevance, diversity, and cold exposure overall and on eligible impressions, then examine the policy's primary relevance metric across cold/warm/mixed clicked-article segments. These are held-out point estimates; aggregate differences alone do not establish statistical significance.

Reuse raw values and population counts already computed in Section 3. No lambda selection, ranking, metric calculation, or file loading is performed here. Preserve unavailable values as missing, keep the units explicit, and omit invariant original-score AUC from improvement plots. Retain the two figure objects and captions for 4.6's final exports; display the new visual comparisons without reprinting the source tables.

#### 4.3.1 Define consistent paired-bar and value formatting

Use blue for baseline and orange for the fixed MMR result. Draw paired bars with value labels and an absolute-delta annotation under each group. Cold-share values are percentages and their deltas are **percentage points**; all other metrics retain raw units. Similarity can be negative, so its axis keeps the signed range.

The small drawing helper is shared by both figures to keep formatting consistent. A missing value receives no fabricated zero bar; a group missing both values is marked `N/A`. Genuine numeric zeros retain their zero value labels.

Nonzero values that would round to zero use scientific notation in displayed text. This changes displayed text only; stored metric values retain their precision.

In [ ]:
# --- 4.3.1: Share paired-bar styling and raw-unit/percentage-point value formatting. ---

def format_heldout_value(metric, value, delta=False):
    """Format chart text only; keep all source metric values at their original precision."""
    if pd.isna(value):
        return "N/A"
    # Cold share uses percentages/points; other metrics retain their raw units.
    cold_share = metric == "cold_share"
    scaled_value = 100 * value if cold_share else value
    digits = max(0, REPORT_DISPLAY_DECIMALS - 2) if cold_share else REPORT_DISPLAY_DECIMALS
    suffix = (" pp" if delta else "%") if cold_share else ""
    text = f"{scaled_value:+.{digits}f}" if delta else f"{scaled_value:.{digits}f}"
    # Show small nonzero values explicitly instead of rounding them to signed zero.
    if scaled_value != 0 and float(text) == 0:
        text = f"{scaled_value:+.2e}" if delta else f"{scaled_value:.2e}"
    return text + suffix


def draw_heldout_pairs(axis, rows, group_labels, metric):
    """Draw existing baseline/MMR values in the supplied scope/segment order."""
    positions = np.arange(len(rows))
    width = 0.28
    pair_offset = 0.19  # A small gap keeps neighbouring value labels visually separate.
    for column, offset, color in [
        ("baseline", -pair_offset, REPORT_COLORS["baseline"]),
        ("mmr", pair_offset, REPORT_COLORS["mmr"]),
    ]:
        bars = axis.bar(positions + offset, rows[column], width, color=color, zorder=3)
        # Empty labels accompany missing bars; actual zero values remain explicitly labelled.
        labels = ["" if pd.isna(value) else format_heldout_value(metric, value) for value in rows[column]]
        axis.bar_label(bars, labels=labels, padding=3, fontsize=8)
    axis.set_xticks(positions, [
        f"{label}\nDelta {format_heldout_value(metric, change, delta=True)}"
        for label, change in zip(group_labels, rows["delta"])
    ], fontsize=8)
    for position, missing in zip(positions, rows[["baseline", "mmr"]].isna().all(axis=1)):
        if missing:
            axis.text(position, 0.12, "N/A", transform=axis.get_xaxis_transform(),
                      ha="center", va="bottom", fontsize=9, color="0.4")
    # Explicit category bounds retain label spacing when NaN bars have no data extent.
    axis.set_xlim(-0.6, len(rows) - 0.4)
    axis.grid(False, axis="x")
    axis.margins(y=0.22)
    if metric == "cold_share":
        axis.yaxis.set_major_formatter(PercentFormatter(xmax=1))
    else:
        axis.ticklabel_format(axis="y", style="plain", useOffset=False)


# A shared legend works even when a particular panel has no available metric values.
heldout_legend_handles = [
    Patch(facecolor=REPORT_COLORS["baseline"], label="Baseline"),
    Patch(facecolor=REPORT_COLORS["mmr"], label=f"MMR (lambda = {evaluation_settings['lambda']:g})"),
]

#### 4.3.2 Compare overall and eligible-slate performance

Use six panels for MRR, nDCG@5, nDCG@10, unique categories, signed intra-list similarity, and cold share. Each panel uses its own units/scale. Show both overall and eligible populations with the same baseline/MMR colour convention and show absolute changes beneath the corresponding bars.

Overall relevance uses all impressions, including short/exact-k slates whose ranks MMR leaves unchanged. Overall diversity uses only full top-k lists; exact-k lists are included there but are ineligible for reranking. The caption identifies valid/skipped relevance impressions and used/excluded diversity lists for both scopes. Higher relevance/category variety and lower similarity have different interpretations; cold share describes exposure.

In [ ]:
# --- 4.3.2: Display six held-out metric panels without recalculating the evaluation. ---
evaluation_comparison_fig, evaluation_comparison_axes = plt.subplots(
    2, 3, figsize=FIGURE_SIZES["evaluation"],
)
for axis, metric in zip(evaluation_comparison_axes.flat, REPORT_METRIC_ORDER):
    # Reindex the small view to keep overall/eligible groups consistent across every panel.
    rows = evaluation_plot_data.loc[evaluation_plot_data["metric"].eq(metric)].set_index("report_scope").reindex(REPORT_SCOPE_ORDER)
    draw_heldout_pairs(axis, rows, ["Overall", "Eligible"], metric)
    if metric != "intra_list_sim":
        axis.set_ylim(bottom=0)
    axis.set_title(METRIC_LABELS[metric])
    if metric in ("mrr", "ndcg@5", "ndcg@10"):
        axis.set_ylabel("Score\n(higher is better)")
    elif metric == "unique_categories":
        axis.set_ylabel("Categories per list\n(higher is more varied)")
    elif metric == "intra_list_sim":
        axis.set_ylabel("Signed cosine\n(lower is more diverse)")
    else:
        axis.set_ylabel("Displayed articles (%)")
evaluation_comparison_fig.suptitle("Held-out relevance, diversity, and cold exposure", y=0.98)
evaluation_comparison_fig.legend(handles=heldout_legend_handles, loc="lower center",
                                 bbox_to_anchor=(0.5, 0.12), ncol=2, frameon=False)
evaluation_comparison_fig.subplots_adjust(left=0.08, right=0.98, top=0.86, bottom=0.27,
                                         hspace=0.70, wspace=0.40)

# Use the retained metric populations; the same candidate/impression sets underlie both rankings.
all_relevance_population = eval_baseline_relevance.loc[("all_impressions", "all")]
eligible_relevance_population = eval_baseline_relevance.loc[("eligible_impressions", "all")]
all_diversity_population = eval_baseline_diversity.loc["all_full_topk_lists"]
eligible_diversity_population = eval_baseline_diversity.loc["eligible_impressions"]
evaluation_comparison_caption = (
    f"Relevance valid/skipped: overall {int(all_relevance_population['n_impressions']):,}/{int(all_relevance_population['n_skipped']):,}; "
    f"eligible {int(eligible_relevance_population['n_impressions']):,}/{int(eligible_relevance_population['n_skipped']):,}.\n"
    f"Diversity full/excluded-short lists: overall {int(all_diversity_population['n_lists']):,}/{int(all_diversity_population['n_short_lists_excluded']):,}; "
    f"eligible {int(eligible_diversity_population['n_lists']):,}/{int(eligible_diversity_population['n_short_lists_excluded']):,}.\n"
    f"Eligible means > {evaluation_settings['top_k']} candidates. Delta = MMR - baseline; cold-share deltas use percentage points."
)
evaluation_comparison_fig.text(0.02, 0.02, evaluation_comparison_caption, ha="left", va="bottom", fontsize=8)
plt.show()

#### 4.3.3 Compare clicked-article segment relevance

Plot the policy's primary relevance metric for cold, warm, and mixed clicked-article segments. Use separate all-impression and eligible-impression panels with a shared y-axis, retaining the same baseline/MMR colours. Segment labels include valid (`n`) and skipped impression counts, plus the absolute metric change.

The segment describes clicked articles within each complete impression, not a filtered candidate pool or a subset of cold candidates. Missing segments remain `N/A`; small counts remain visible beside their estimates. These held-out differences describe the fixed choice and do not revise the lambda or selection policy.

In [ ]:
# --- 4.3.3: Display the existing cold/warm/mixed estimates and their population counts. ---
evaluation_segments_fig, evaluation_segments_axes = plt.subplots(
    1, 2, figsize=FIGURE_SIZES["segments"], sharey=True,
)
for axis, scope in zip(evaluation_segments_axes, ["all_impressions", "eligible_impressions"]):
    rows = segment_plot_data.loc[segment_plot_data["scope"].eq(scope)].set_index("segment").reindex(SEGMENT_ORDER)
    # Counts describe the complete impressions used by each metric, including exclusions.
    group_labels = [
        f"{segment.capitalize()}\nn = {int(row['n_impressions']):,}\nskipped = {int(row['n_skipped']):,}"
        for segment, row in rows.iterrows()
    ]
    draw_heldout_pairs(axis, rows, group_labels, report_relevance_metric)
    axis.set_title(REPORT_SCOPE_LABELS[scope])
# Resolve the common range after both panels are drawn, so higher eligible values stay visible.
evaluation_segments_axes[0].set_ylim(bottom=0)
evaluation_segments_axes[0].set_ylabel(f"{METRIC_LABELS[report_relevance_metric]} (higher is better)")
evaluation_segments_fig.suptitle(f"Held-out clicked-article segment relevance: {METRIC_LABELS[report_relevance_metric]}", y=0.98)
evaluation_segments_fig.legend(handles=heldout_legend_handles, loc="lower center",
                               bbox_to_anchor=(0.5, 0.13), ncol=2, frameon=False)
evaluation_segments_fig.subplots_adjust(left=0.08, right=0.98, top=0.80, bottom=0.39, wspace=0.18)
evaluation_segments_caption = (
    "Segments use clicked-article cold status within full slates. n = valid impressions; skipped impressions are excluded from relevance means.\n"
    "Delta = MMR - baseline in raw metric units. N/A indicates an unavailable estimate; small segments require cautious interpretation."
)
evaluation_segments_fig.text(0.02, 0.02, evaluation_segments_caption, ha="left", va="bottom", fontsize=8)
plt.show()

### 4.4 Inspect reproducible illustrative recommendation slates

Select a small set of eligible held-out impressions using the recorded seed and a sorted impression-ID pool. Selection does not depend on labels, score changes, diversity gains, or whether a list changed. The examples illustrate the fixed MMR mechanism; they are not evidence of average performance or a representative estimate of its benefits.

Show the union of each example's baseline/MMR top-k articles once, with titles, original scores, category/cold metadata, clicked labels, both ranks, and movement status. Compute list-level category variety, signed cosine similarity, and cold share only for these few examples. Reuse the existing complete rankings/catalog; do not rerun MMR or aggregate evaluation. Retain raw example tables and selection metadata for 4.6's exports without saving files here.

#### 4.4.1 Select examples independently of their outcomes

Uniformly sample up to three eligible impressions without replacement from sorted IDs, using `evaluation_settings["seed"]`. Sorting makes selection independent of input row order. Use all available impressions if fewer than three are eligible. If none are eligible, retain empty example tables and report that no eligible examples exist.

Keep the selected IDs and method in `example_selection_metadata`. The example count is a presentation choice; it does not influence tuning, evaluation, or which articles MMR selects.

In [ ]:
# --- 4.4.1: Select a small, reproducible set of eligible impressions before inspecting outcomes. ---
EXAMPLE_COUNT = 3
EXAMPLE_METADATA_PATH = Path(config.PROCESSED_DIR) / "news_metadata.parquet"
example_pool = eval_baseline_slates.index[eval_baseline_slates["rerank_eligible"]].sort_values()
example_impression_ids = np.random.default_rng(evaluation_settings["seed"]).choice(
    example_pool.to_numpy(), size=min(EXAMPLE_COUNT, len(example_pool)), replace=False,
).tolist()
# Record selection provenance for the later report; these are ordinary metadata values.
example_selection_metadata = {
    "method": "uniform sample without replacement from sorted eligible impression IDs",
    "seed": int(evaluation_settings["seed"]), "requested_count": EXAMPLE_COUNT,
    "eligible_pool_size": len(example_pool), "impression_ids": list(example_impression_ids),
    "article_metadata_path": str(EXAMPLE_METADATA_PATH),
}
if example_impression_ids:
    print("Selected illustrative impression IDs:", example_impression_ids)
else:
    print("No eligible impressions are available for example slates.")

#### 4.4.2 Collect displayed articles, titles, and rank movements

Take only articles appearing in the baseline or MMR top-k for the chosen impressions. Both rank columns come from the existing complete reranked table; the original model score, label, category, and cold flag are preserved. A positive `rank_change = baseline_rank - rerank_rank` means movement upward.

Read only the metadata's confirmed `news_id`/`title` columns, filtering to the displayed article IDs, and join titles by ID. Missing titles are labelled unavailable without dropping articles or altering model inputs. Classify articles as entering/leaving the top-k, moving within it, or unchanged. Keep one raw row per example/article; a rank greater than k means that article is outside that ranking's displayed list.

In [ ]:
# --- 4.4.2: Build the small union of baseline/MMR displayed articles and attach titles. ---
example_slates = eval_reranked.loc[
    eval_reranked["impression_id"].isin(example_impression_ids)
    & (eval_reranked["baseline_rank"].le(evaluation_settings["top_k"])
       | eval_reranked["rerank_rank"].le(evaluation_settings["top_k"])),
    KEY_COLUMNS + ["score", "label", COLD_COLUMN, "category", "baseline_rank", "rerank_rank"],
].copy()
if not example_slates.empty:
    # Project/filter the existing metadata for annotation only; category/cold values stay upstream's.
    example_titles = pd.read_parquet(
        EXAMPLE_METADATA_PATH, columns=["news_id", "title"],
        filters=[("news_id", "in", example_slates["news_id"].unique().tolist())], engine="pyarrow",
    )
    example_slates = example_slates.merge(example_titles, on="news_id", how="left", validate="many_to_one")
    example_slates["title"] = example_slates["title"].fillna("[Title unavailable]")
    del example_titles
else:
    example_slates["title"] = pd.Series(dtype="string")

example_slates["example_number"] = example_slates["impression_id"].map(
    {impression_id: number for number, impression_id in enumerate(example_impression_ids, start=1)}
)
example_slates["rank_change"] = example_slates["baseline_rank"] - example_slates["rerank_rank"]
example_slates["in_baseline_top_k"] = example_slates["baseline_rank"].le(evaluation_settings["top_k"])
example_slates["in_mmr_top_k"] = example_slates["rerank_rank"].le(evaluation_settings["top_k"])
# Entry/exit take precedence over within-list movement; flags describe displayed membership only.
in_both_lists = example_slates["in_baseline_top_k"] & example_slates["in_mmr_top_k"]
example_slates["top_k_change"] = np.select(
    [~example_slates["in_baseline_top_k"] & example_slates["in_mmr_top_k"],
     example_slates["in_baseline_top_k"] & ~example_slates["in_mmr_top_k"],
     in_both_lists & example_slates["rank_change"].gt(0),
     in_both_lists & example_slates["rank_change"].lt(0)],
    ["Entered top-k", "Left top-k", "Moved up", "Moved down"], default="Unchanged",
)
example_slates = example_slates.sort_values(["example_number", "baseline_rank"], kind="stable").reset_index(drop=True)

#### 4.4.3 Measure list changes for the selected examples

For each example, compare its top-k order and count entries, exits, and rank movements among retained articles. Measure unique categories, mean pairwise signed cosine, and cold share using the existing raw list-metric helper. These calculations involve only the selected example lists; they do not replace or recompute the held-out aggregate results.

Retain `example_summary` for selection/churn counts and `example_list_metrics` for raw before/after values and absolute deltas. Pairwise similarity is unavailable for k = 1, rather than zero. Unchanged examples, including a selected lambda of 1, remain valid examples and are not replaced with more favorable ones.

In [ ]:
# --- 4.4.3: Measure only the chosen example lists using the established metric definitions. ---
example_summary_rows, example_metric_rows = [], []
for impression_id in example_impression_ids:
    articles = example_slates.loc[example_slates["impression_id"].eq(impression_id)]
    before = articles.loc[articles["in_baseline_top_k"]].sort_values("baseline_rank")
    after = articles.loc[articles["in_mmr_top_k"]].sort_values("rerank_rank")
    example_summary_rows.append({
        "impression_id": impression_id,
        "example_number": int(articles["example_number"].iloc[0]),
        "n_candidates": int(eval_baseline_slates.loc[impression_id, "n_candidates"]),
        "top_k_order_changed": before["news_id"].tolist() != after["news_id"].tolist(),
        "n_entered": int(articles["top_k_change"].eq("Entered top-k").sum()),
        "n_left": int(articles["top_k_change"].eq("Left top-k").sum()),
        "n_moved_within_top_k": int(articles["top_k_change"].isin(["Moved up", "Moved down"]).sum()),
    })
    # Each call measures one full displayed list, using the same category/vector/cold definitions.
    before_metrics = p6_topk_diversity_raw(
        before, category_col="category", emb_lookup=emb_lookup, cold_col=COLD_COLUMN,
        k=evaluation_settings["top_k"],
    )
    after_metrics = p6_topk_diversity_raw(
        after, category_col="category", emb_lookup=emb_lookup, cold_col=COLD_COLUMN,
        k=evaluation_settings["top_k"],
    )
    for metric in ["unique_categories", "intra_list_sim", "cold_share"]:
        baseline_value = before_metrics.get(metric, np.nan)
        mmr_value = after_metrics.get(metric, np.nan)
        example_metric_rows.append({"impression_id": impression_id, "metric": metric,
                                    "baseline": baseline_value, "mmr": mmr_value,
                                    "delta": mmr_value - baseline_value})
# Explicit columns keep the exportable empty-table schema when the eligible pool is empty.
example_summary = pd.DataFrame(example_summary_rows, columns=[
    "impression_id", "example_number", "n_candidates", "top_k_order_changed",
    "n_entered", "n_left", "n_moved_within_top_k",
]).set_index("impression_id")
example_list_metrics = pd.DataFrame(example_metric_rows, columns=[
    "impression_id", "metric", "baseline", "mmr", "delta",
]).set_index(["impression_id", "metric"])
del example_summary_rows, example_metric_rows

#### 4.4.4 Inspect one combined before/after table per example

Display the union in baseline order, with both complete ranks and entry/exit/movement labels. The baseline top-k consists of rows with baseline rank <= k; the MMR top-k consists of rows with MMR rank <= k, ordered by MMR rank. This avoids printing an unchanged article list twice while preserving both orders explicitly.

Below each article table, show the new per-example diversity/exposure comparison. Use the existing chart formatter so cold-share deltas appear in percentage points, while raw exported values remain fractions. Titles may be shortened by the display settings; `example_slates` retains the full strings. Click labels are annotations, not inputs to MMR or example selection. Use these observations to explain mechanisms alongside the aggregate evidence, without claiming that sampled examples demonstrate average gains or fairness.

In [ ]:
# --- 4.4.4: Display each example once, with both orders and its local list-metric changes. ---
from IPython.display import display

for impression_id in example_impression_ids:
    summary = example_summary.loc[impression_id]
    print(f"Example {int(summary['example_number'])}: impression {impression_id} "
          f"({int(summary['n_candidates']):,} candidates)")
    print(f"Entered: {int(summary['n_entered'])}; left: {int(summary['n_left'])}; "
          f"moved within top-k: {int(summary['n_moved_within_top_k'])}.")
    articles = example_slates.loc[example_slates["impression_id"].eq(impression_id), [
        "news_id", "title", "category", COLD_COLUMN, "label", "score",
        "baseline_rank", "rerank_rank", "rank_change", "top_k_change",
    ]]
    # Display precision/title width affect only this view; raw titles/scores remain unchanged.
    with pd.option_context("display.max_colwidth", 80, "display.max_rows", None,
                           "display.max_columns", None, "display.precision", REPORT_DISPLAY_DECIMALS):
        display(articles)
    comparison = example_list_metrics.xs(impression_id, level="impression_id").copy()
    for column in ["baseline", "mmr", "delta"]:
        comparison[column] = [
            format_heldout_value(metric, value, delta=(column == "delta"))
            for metric, value in comparison[column].items()
        ]
    display(comparison.rename(index=METRIC_LABELS))

### 4.5 Write evidence-based conclusions

Connect the tuning-only decision, held-out metric changes, clicked-article segments, and seeded examples into one concise report. Use the recorded policy/settings and existing summaries. Do not rerun selection, rankings, metrics, or statistical analyses, and do not assume that MMR improved relevance, diversity, or cold exposure.

The generated conclusion distinguishes numerical directions from claims of benefit, keeps overall/eligible denominators explicit, and describes unavailable estimates honestly. Display the narrative once rather than printing the source tables again. Retain `report_conclusions_markdown` for the final export step in 4.6; any later discussion of article content should be grounded in the actual 4.4 examples.

#### 4.5.1 Explain the recorded tuning decision

Describe the selected model, lambda, scaling, primary relevance metric, relative loss budget, diversity objective, and policy scope. Use the saved decision row to state the selected point's tuning relevance versus lambda = 1. Explain optional cold/segment constraints in their correct units and identify the choice as the best qualifying point among the tested grid, not a globally optimal lambda.

The formatting helper below turns existing metric rows into sentences. It uses the same display precision and percentage-point convention as the held-out figures and leaves the raw values untouched. These are presentation transformations, not new evaluation calculations.

In [ ]:
# --- 4.5.1: Explain the saved tuning choice and share factual comparison-sentence formatting. ---
def describe_report_comparison(metric, values):
    """Describe stored baseline/MMR values without inventing unavailable results."""
    if pd.isna(values["baseline"]) or pd.isna(values["mmr"]):
        return f"{METRIC_LABELS[metric]} is unavailable for this population."
    # Reuse chart formatting: raw metric deltas, except percentage points for cold share.
    return (f"{METRIC_LABELS[metric]}: {format_heldout_value(metric, values['baseline'])} before versus "
            f"{format_heldout_value(metric, values['mmr'])} after MMR "
            f"(change {format_heldout_value(metric, values['delta'], delta=True)}).")


report_decision = selected_tuning_metrics["decision_row"]
report_constraint_descriptions = []
if "min_cold_share_delta" in report_policy["constraints"]:
    report_constraint_descriptions.append(
        f"minimum cold-share change {100 * report_policy['constraints']['min_cold_share_delta']:+g} percentage points"
    )
for segment, limit in report_policy["constraints"].get("max_segment_relative_relevance_loss", {}).items():
    report_constraint_descriptions.append(f"maximum {segment}-segment relative relevance loss {100 * limit:g}%")
report_constraint_sentence = (
    "Additional tuning constraints: " + "; ".join(report_constraint_descriptions) + "."
    if report_constraint_descriptions else "The policy used only the primary relevance-loss and diversity criteria."
)
# Explain the existing decision; do not reassess qualification using held-out data.
report_diversity_goal = "minimize" if report_diversity_metric == "intra_list_sim" else "maximize"
report_tuning_conclusion = (
    f"The tuning policy selected lambda = {evaluation_settings['lambda']:g} from "
    f"{len(selected_tuning_settings['lambdas'])} tested settings to {report_diversity_goal} "
    f"{METRIC_LABELS[report_diversity_metric].lower()}, allowing at most "
    f"{100 * report_policy['max_relative_relevance_loss']:g}% relative loss in {METRIC_LABELS[report_relevance_metric]}. "
    f"Its tuning relevance was {report_decision['primary_relevance']:.{REPORT_DISPLAY_DECIMALS}f}, "
    f"versus {report_decision['reference_relevance']:.{REPORT_DISPLAY_DECIMALS}f} at lambda = 1. "
    f"{report_scope_note}. {report_constraint_sentence} "
    "This identifies the chosen operating point among the tested grid, not a globally optimal parameter. "
    "The lambda and policy were fixed before held-out evaluation (tuning figures: Section 4.2)."
)

#### 4.5.2 Connect held-out changes, segment evidence, and examples

Describe the primary relevance metric, selected diversity objective, and cold exposure in both reporting scopes, alongside the existing valid/skipped and full-list counts. Summarize the direction of the primary trade-off only when both estimates exist; otherwise state that it cannot be fully assessed for that population. Lower cosine and higher category count have different meanings, and cold exposure is not itself a fairness result.

Use the cold/warm/mixed primary-scope estimates and counts already prepared in 3.6. For the seeded examples, report how many displayed top-k orders changed and identify the impressions inspected in 4.4. The example set illustrates behavior; its change rate is not an estimate of the population-wide effect. No aggregate inference is drawn from the few sampled lists.

In [ ]:
# --- 4.5.2: Build balanced held-out/segment/example paragraphs from existing results. ---
report_heldout_paragraphs = []
for name, relevance_scope, diversity_scope in [
    ("Overall", "all_impressions", "all_full_topk_lists"),
    ("Eligible impressions", "eligible_impressions", "eligible_impressions"),
]:
    relevance = evaluation_results.loc[("relevance", relevance_scope, "all", report_relevance_metric)]
    diversity = evaluation_results.loc[("diversity", diversity_scope, "list", report_diversity_metric)]
    exposure = evaluation_results.loc[("diversity", diversity_scope, "list", "cold_share")]
    relevance_counts = eval_baseline_relevance.loc[(relevance_scope, "all")]
    diversity_counts = eval_baseline_diversity.loc[diversity_scope]
    report_heldout_paragraphs.append(
        f"**{name}.** {describe_report_comparison(report_relevance_metric, relevance)} "
        f"{describe_report_comparison(report_diversity_metric, diversity)} "
        f"{describe_report_comparison('cold_share', exposure)} "
        f"Valid/skipped relevance impressions: {int(relevance_counts['n_impressions']):,}/"
        f"{int(relevance_counts['n_skipped']):,}; full/excluded-short diversity lists: "
        f"{int(diversity_counts['n_lists']):,}/{int(diversity_counts['n_short_lists_excluded']):,}."
    )
# Interpret directions only for available estimates, without treating small changes as significant.
if pd.notna(eval_primary_relevance["delta"]) and pd.notna(eval_primary_diversity["delta"]):
    relevance_direction = "higher" if eval_primary_relevance["delta"] > 0 else "lower" if eval_primary_relevance["delta"] < 0 else "unchanged"
    diversity_direction = "higher" if eval_primary_diversity["delta"] > 0 else "lower" if eval_primary_diversity["delta"] < 0 else "unchanged"
    report_tradeoff_assessment = (
        f"In the tuning-declared reporting scope, held-out point estimates showed {relevance_direction} "
        f"{METRIC_LABELS[report_relevance_metric]} and {diversity_direction} "
        f"{METRIC_LABELS[report_diversity_metric].lower()}."
    )
else:
    report_tradeoff_assessment = "The primary held-out trade-off cannot be fully assessed because at least one estimate is unavailable in the declared reporting scope."
report_tradeoff_assessment += (
    " Lower cosine denotes greater semantic diversity; more unique categories denotes greater category variety. "
    "Cold share measures exposure, not demonstrated fairness or cold-article relevance. "
    "The held-out figures are in Section 4.3."
)
report_segment_bullets = [
    f"- **{segment.capitalize()}:** {describe_report_comparison(report_relevance_metric, values)} "
    f"Valid: {int(values['n_impressions']):,}; skipped: {int(values['n_skipped']):,}."
    for segment, values in eval_primary_segments.iterrows()
]
if not example_summary.empty:
    report_example_conclusion = (
        f"The seed-selected illustration set contained {len(example_summary)} eligible impressions; "
        f"{int(example_summary['top_k_order_changed'].sum())} had a different top-{evaluation_settings['top_k']} order. "
        f"Inspected IDs: {', '.join(str(value) for value in example_impression_ids)}. "
        "Section 4.4 shows their article movements and local category/similarity/exposure changes. "
        "These examples illustrate the mechanism and do not estimate average effects."
    )
else:
    report_example_conclusion = "No example slates were selected for illustration; no qualitative conclusions are drawn from examples."

#### 4.5.3 Display the conclusion and retain it for export

Assemble one Markdown report containing the run identity, tuning rationale, both held-out scopes, primary-scope segment results, example context, and the established interpretation limits. This is a factual draft for the report/presentation; content-specific explanation should refer to the actual article titles and movement tables, rather than being inferred automatically from category or cosine changes.

The narrative is displayed here once. `report_conclusions_markdown` remains in memory for writing to `REPORT_CONCLUSIONS_PATH` in 4.6. Existing result tables, findings, settings, and limitations are not modified, and no files are written in this subsection.

In [ ]:
# --- 4.5.3: Display a factual conclusion once, retaining the same Markdown for final export. ---
from IPython.display import Markdown

# Reuse the existing limits and population conventions rather than creating another assessment policy.
report_conclusions_markdown = (
    "# MMR reranking: evidence-based conclusions\n\n"
    f"**Model:** {evaluation_settings['model']} | **Held-out date:** {evaluation_settings['evaluation_date']} | "
    f"**Lambda:** {evaluation_settings['lambda']:g} | **Top k:** {evaluation_settings['top_k']} | "
    f"**Scaling:** `{evaluation_settings['relevance_scaling']}` | **Seed:** {evaluation_settings['seed']}\n\n"
    "## Tuning decision\n\n" + report_tuning_conclusion + "\n\n"
    "## Held-out observations\n\n" + "\n\n".join(report_heldout_paragraphs) + "\n\n"
    + report_tradeoff_assessment + "\n\n"
    f"## Clicked-article segments: {REPORT_SCOPE_LABELS[report_policy['relevance_scope']]}\n\n"
    + "\n".join(report_segment_bullets) + "\n\n"
    "## Illustrative slates\n\n" + report_example_conclusion + "\n\n"
    "## Interpretation limits\n\n" + "\n".join(f"- {limit}" for limit in evaluation_limitations)
)
display(Markdown(report_conclusions_markdown))

### 4.6 Export the completed report artifacts

Save the existing unrounded tuning/evaluation results, tuning selection evidence, illustrative example tables, four figures in PNG/PDF, and the conclusion from 4.5. Record the fixed experiment settings, captions, software versions, example-selection method, and artifact locations in one JSON metadata file so the exported CSVs remain interpretable without notebook state.

Reuse Section 3.6's detailed evaluation CSVs and `run.json`; do not rewrite those files here. This step performs ordinary exports only: no data loading, selection, reranking, metric calculation, or repeated result display. Fixed filenames are replaced on the next complete run, so copy the output folder elsewhere before running a different experiment if both runs must be retained.

The full per-lambda Step 6 score Parquets are written in 3.4.3 and listed in the inherited `step6_handoff` metadata; this section does not rerun their rankings or rewrite those files.

#### 4.6.1 Save result tables and illustrative examples

Write the two configured top-level result CSVs, the tuning budget comparison and complete qualification/selection table, and the three example tables. Preserve named index levels for metric tables and omit the artificial row index for article rows. Use normal CSV numeric precision; missing estimates remain blank rather than becoming zeros. Empty example tables still export their column headers.

In [ ]:
# --- 4.6.1: Export completed tables at their stored numeric precision. ---
REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Each entry pairs a table with its destination and whether its named index carries data.
report_table_exports = {
    "tuning_results": (tuning_results, TUNING_RESULTS_PATH, True),
    "evaluation_results": (evaluation_results, EVAL_RESULTS_PATH, True),
    "tuning_budget_comparison": (tuning_budget_comparison, REPORT_DIR / "tuning_budget_comparison.csv", True),
    "tuning_selection": (tuning_selection_table, REPORT_DIR / "tuning_selection.csv", True),
    "example_slates": (example_slates, EXAMPLE_SLATES_PATH, False),
    "example_summary": (example_summary, REPORT_DIR / "example_summary.csv", True),
    "example_list_metrics": (example_list_metrics, REPORT_DIR / "example_list_metrics.csv", True),
}
report_table_manifest = {}
for name, (table, path, include_index) in report_table_exports.items():
    table.to_csv(path, index=include_index, encoding="utf-8")
    # CSV does not retain DataFrame attrs, so preserve index conventions in the JSON manifest.
    report_table_manifest[name] = {
        "path": str(path), "rows": len(table),
        "index_columns": list(table.index.names) if include_index else [],
    }

#### 4.6.2 Save the existing figures and captions

Export each figure already displayed in 4.2 and 4.3 as a 300-DPI PNG and a vector PDF using the configured style and formats. Retain captions and file locations in the metadata rather than duplicating plots or rebuilding them. Tight bounding boxes include labels and legends outside the axes.

In [ ]:
# --- 4.6.2: Save retained figure objects without recreating plots or results. ---
report_figure_exports = {
    "tuning_tradeoff": (tuning_tradeoff_fig, tuning_tradeoff_caption),
    "tuning_diagnostics": (tuning_diagnostics_fig, tuning_diagnostics_caption),
    "evaluation_comparison": (evaluation_comparison_fig, evaluation_comparison_caption),
    "evaluation_segments": (evaluation_segments_fig, evaluation_segments_caption),
}
report_figure_manifest = {}
for name, (figure, caption) in report_figure_exports.items():
    paths = {}
    for file_format in FIGURE_FORMATS:
        path = FIGURE_DIR / f"{name}.{file_format}"
        figure.savefig(path, format=file_format, dpi=FIGURE_EXPORT_DPI, bbox_inches="tight")
        paths[file_format] = str(path)
    report_figure_manifest[name] = {"files": paths, "caption": caption}

#### 4.6.3 Save the conclusion and reproducibility metadata

Write the Markdown conclusion exactly as displayed in 4.5. Extend the existing evaluation metadata with the tuning settings, example provenance, presentation settings, and a manifest of the new files plus the detailed outputs already written in 3.6. Keep missing numeric estimates in CSVs; the JSON contains descriptive metadata and settings and uses strict JSON serialization.

Paths identify the configured inputs and outputs; they are not content hashes or a snapshot of upstream files. Reproducing the experiment still requires the same upstream artifacts, dependencies, and recorded settings. The inherited `step6_handoff` includes all per-lambda score file paths/counts and the independent-diversity evaluation requirement. The metadata records Matplotlib alongside the previously recorded Python/NumPy/pandas versions.

In [ ]:
# --- 4.6.3: Export the displayed narrative and a self-describing artifact manifest. ---
REPORT_CONCLUSIONS_PATH.write_text(report_conclusions_markdown + "\n", encoding="utf-8")
# Reuse Section 3.6's completed summaries; record their locations rather than writing duplicates.
report_existing_evaluation_files = {
    name: str(evaluation_summary_dir / name)
    for name in ("metrics.csv", "relevance.csv", "diversity.csv", "overview.csv", "segments.csv", "run.json")
}
report_metadata = {
    **deepcopy(evaluation_summary_record),
    "tuning_settings": deepcopy(selected_tuning_settings),
    "example_selection": deepcopy(example_selection_metadata),
    "presentation": {
        "display_decimals": REPORT_DISPLAY_DECIMALS,
        "figure_dpi": FIGURE_EXPORT_DPI, "figure_formats": list(FIGURE_FORMATS),
        "figure_sizes": deepcopy(FIGURE_SIZES), "style": deepcopy(REPORT_STYLE),
        "colors": deepcopy(REPORT_COLORS),
    },
    "export_conventions": {
        "csv_precision": "stored numeric precision; no display rounding applied",
        "missing_csv_values": "blank fields denote unavailable values, not zero",
        "cold_share_units": "CSV fractions; displayed changes in percentage points",
        "overwrite": "fixed destinations are replaced on each complete run",
        "provenance": "configured paths and identifiers; no upstream content hashes",
    },
    "artifacts": {
        "tables": report_table_manifest, "figures": report_figure_manifest,
        "conclusions": str(REPORT_CONCLUSIONS_PATH), "metadata": str(REPORT_METADATA_PATH),
        "existing_evaluation_summary": report_existing_evaluation_files,
    },
}
report_metadata["software_versions"]["matplotlib"] = matplotlib.__version__
# Serialize settings explicitly because CSV attrs and figure captions are otherwise lost.
with REPORT_METADATA_PATH.open("w", encoding="utf-8") as metadata_file:
    json.dump(report_metadata, metadata_file, indent=2, ensure_ascii=False, allow_nan=False)

#### 4.6.4 Report the saved destinations

Print only the new export counts and locations. The metric tables, examples, figures, and conclusions have already been displayed upstream; this final message confirms where to find the reusable artifacts.

In [ ]:
# --- 4.6.4: Confirm the completed exports without repeating experiment results. ---
print(f"Saved {len(report_table_manifest)} CSV tables, "
      f"{sum(len(entry['files']) for entry in report_figure_manifest.values())} figure files, "
      "conclusions.md, and report_metadata.json.")
print("Top-level results:", TUNING_RESULTS_PATH, "and", EVAL_RESULTS_PATH)
print("Report artifacts:", REPORT_DIR)
print("Detailed evaluation summaries from Section 3.6:", evaluation_summary_dir)